# v4 REVİZYON NOTU — makale revizyon döngüsü sonrası

Bu notebook, `MAMBA.ipynb` (v3) dosyasının **yamalanmış** hâlidir; v3'ün hiçbir
hücresi, aşağıda listelenen değişiklikler dışında değiştirilmemiştir.

| Hücre | Değişiklik | Gerekçe |
|---|---|---|
| 3 | CICIDS2018 **ilan edilmiş negatif kontrol** olarak belgelendi | 79 özelliğin 15'i `seq_len=64` ile kesiliyor; düşük başarım beklenir ve önceden ilan edilir |
| 6 | Cache'ten yüklenen veri setleri için bölme stratejisi **ham veriden yeniden türetilip doğrulanıyor** | v3'te cache kullanıldığında `SPLIT_STRATEGY_LOG` boş kalıyor, garanti iddia edilip kanıtlanmıyordu |
| **6c (YENİ)** | Tek-eşik ayrılabilirlik denetimi | Tek bir özellik sınıfları ayırıyorsa, o kıyaslama mimarileri ayırt edemez. Çoğunluk sınıfı taban çizgisi de raporlanır |
| **7c (YENİ)** | S6 uygulama doğrulaması (CHECK 1/2/3) | Ana bulgu, yeniden uygulanan tek bileşen hakkında; doğruluğu varsayılmaz, ölçülür |
| 11 | `paired_significance_test` → Cohen'in **d_z** etki büyüklüğü | p-değeri etki büyüklüğü ile tekrar sayısını karıştırır; d_z tohum sayısından bağımsızdır |
| 12 | Tüm grafik etiketleri **İngilizce**; `disp()` ile BiLSTM adlandırması | Makale İngilizce; yinelemeli baseline çift yönlüdür |
| 13 | Özet tablosuna **makro precision/recall ve AUC** eklendi | Yöntem bölümü bu metrikleri vaat ediyordu |
| 15 | **Önceden belirlenmiş** karşılaştırma modeli (Transformer) birincil; post-hoc "en iyi rakip" ikincil tabloya taşındı | Karşılaştırma modelini testte kullanılan aynı veriye bakarak seçmek p-değerlerini anti-konservatif yapar; Holm bunu düzeltmez |

**Çalıştırma:** v3 ile aynı — GPU zorunlu, `Runtime > Run all`. Yeni 6c ve 7c
hücreleri hafiftir (7c CPU'da birkaç saniye).

**Uyarı:** `data_version` etiketini değiştirmediyseniz mevcut cache ve
checkpoint'ler yeniden kullanılır; bölme yeniden türetme bloğu bunu varsayar.


# Ağ Saldırı Tespit Sistemlerinde Durum-Uzayı Modellerinin (Mamba) Çekişmeli Sağlamlığı
### CNN, LSTM ve Transformer Mimarileriyle Karşılaştırmalı Bir Analiz — Deneysel Kod

**Bilişim Teknolojileri Dergisi (BTD) için hazırlanmıştır.**

Bu notebook, aşağıdaki araştırma sorusunu deneysel olarak yanıtlar:
> Mamba (Selective State Space Model / S6) tabanlı bir Ağ Saldırı Tespit Sistemi
> (NIDS), FGSM, PGD ve kara-kutu transfer saldırıları altında CNN, LSTM ve
> Transformer tabanlı NIDS'lere kıyasla ne ölçüde sağlamdır (robust)? Hangi
> saldırı alt-türleri en çok kırılganlık yaratır ve basit bir savunma
> (adversarial training) bu kırılganlığı ne ölçüde azaltır?

**v3 REVIZYON NOTU (hakem geri bildirimi sonrasi):** Rastgele SATIR-bazli train/test split, GRUP-farkindali (host-cifti/flow, mumkun degilse zaman-bazli) split ile degistirildi (bkz. Hucre 6) -- akis-duzeyinde veri sizintisi riskini azaltir. Coklu (veri-seti x saldiri) istatistiksel karsilastirmaya Holm-Bonferroni duzeltmesi eklendi (bkz. Hucre 11/15, Tablo T2). Checkpoint/cache dosya adlarina `data_version` etiketi eklendi; bu REVIZYONDAN ONCEKI (eski split ile uretilmis) cache/checkpoint dosyalari OTOMATIK OLARAK gecersiz sayilir ve yeniden uretilir.

**Tasarım ilkeleri:**
- ✅ **Hard-GPU-Only**: GPU yoksa çalışma başlamadan durur (Hücre 1).
- ✅ **Ücretsiz/açık-kaynak**: Hiçbir ücretli API kullanılmaz (PyTorch, scikit-learn,
  matplotlib, Gmail SMTP — hepsi ücretsiz).
- ✅ **Kesintisiz & güvenli**: Her aşama checkpoint'lidir; "Run All" tekrar
  çalıştırıldığında tamamlanmış işler atlanır. Her önemli olayda/hatada
  otomatik e-posta bildirimi gönderilir.
- ✅ **"Publication-ready" çıktı**: Tüm grafikler 300dpi PNG + vektör PDF,
  tüm tablolar CSV + LaTeX (booktabs) + Markdown formatında Drive'a kaydedilir.
- ✅ **Veri seti tekrar indirilmez**: Veri setleri Drive'da önceden bulunmalıdır;
  kod sadece varlığını doğrular (istisna: küçük NSL-KDD için tek seferlik otomatik indirme).
- ✅ **İstatistiksel güç**: 7 bağımsız tohum, eşli t-testi + Wilcoxon testi.
- ✅ **Saldırı alt-türü analizi**: Binary'nin ötesinde, DoS/Probe/R2L/U2R/Exploits
  gibi alt-kategoriler bazında kırılganlık kırılımı.
- ✅ **Ölçülen verimlilik**: Parametre sayısının ötesinde, gerçek GPU gecikme/verim ölçümü.
- ✅ **Savunma değerlendirmesi**: FGSM-karışık adversarial training öncesi/sonrası karşılaştırma.
- ✅ **Hiperparametre adaleti ablasyonu**: Sonuçların keyfi hiperparametre seçimine bağlı olmadığının kanıtı.
- ✅ **Sınıf dengesizliğine dayanıklılık**: Ters-frekans ağırlıklı loss + dengelenmiş doğruluk ile model seçimi (gerçek NIDS verisi ~%90+ benign içerir).
- ✅ **Otomatik veri sağlık kontrolü**: NaN/Inf, sınıf dengesizliği, sıfır-varyans özellik ve yetersiz saldırı-kategorisi örneklemi tespiti.
- ✅ **Duman testi (smoke test) modu**: Gerçek veriyle küçük ölçekte ucuz bir ön-doğrulama imkânı (`SMOKE_TEST=True`).
- ✅ **Dinamik bulgu anlatısı**: Ölçülen verimlilik hangi yönde çıkarsa çıksın, makaleye hazır bir yorum otomatik üretilir.

---


## KURULUM (Notebook'u çalıştırmadan önce lütfen okuyun)

### 1. Çalışma zamanı (Runtime)
`Runtime > Change runtime type > Hardware accelerator > GPU` seçin (Colab Pro+ ile
A100 talep edin). Bu notebook **GPU olmadan çalışmayı reddedecek** şekilde tasarlanmıştır.

### 2. Colab Secrets (sol panel → 🔑 anahtar simgesi)
Aşağıdaki üç secret'ı ekleyin ve bu notebook için erişime izin verin:

| Secret adı | Açıklama |
|---|---|
| `GMAIL_USER` | Bildirim gönderecek Gmail adresiniz |
| `GMAIL_APP_PASSWORD` | Google Hesabı → Güvenlik → **Uygulama Şifreleri**'nden üretilen 16 haneli şifre (normal Gmail şifreniz DEĞİL) |
| `GMAIL_TO` | Bildirimlerin gönderileceği adres (kendi adresiniz olabilir) |

> Bu secret'lar eksikse çalışma **yine de devam eder**, sadece e-posta bildirimleri devre dışı kalır.

### 3. Google Drive klasör yapısı — VERİ SETLERİNİ ÖNCEDEN YERLEŞTİRİN
Drive'ınızda şu klasör yapısını oluşturun ve veri setlerini **bir kez, manuel olarak** indirip yerleştirin:

```
MyDrive/
└── BTD_Mamba_NIDS_Robustness/
    └── datasets/
        ├── NSL-KDD/
        │   ├── KDDTrain+.txt      (bulunamazsa kod TEK SEFERLİK otomatik indirmeyi dener)
        │   └── KDDTest+.txt
        ├── NF-UNSW-NB15/
        │   └── NF-UNSW-NB15-v2.csv   ← https://staff.itee.uq.edu.au/marius/NIDS_datasets/ adresinden manuel indirin
        └── CICIDS2018/
            └── CICIDS2018_sample.csv ← https://www.unb.ca/cic/datasets/ids-2018.html adresinden manuel indirin
```

Diğer tüm klasörler (`cache/`, `checkpoints/`, `figures/`, `tables/`, `logs/`, `reports/`, `results/`)
kod tarafından **otomatik olarak** oluşturulur.

**Neden manuel indirme?** NF-UNSW-NB15 ve CICIDS2018 çok büyük (GB mertebesinde) veri
setleridir ve resmi kaynakları zaman zaman token'lı/kayıt gerektiren bağlantılar
kullanır. Bu nedenle kırılgan/güvenilmez otomatik indirme denemeleri yerine —
kullanıcının talebi doğrultusunda — veri setlerinin **önceden Drive'a yerleştirilmesi**
tercih edilmiştir.

### 4. Duman Testi (ÖNERİLİR) → sonra tam çalışma
Bu kod daha önce gerçek Colab GPU'sunda ve gerçek veri setleriyle hiç
çalıştırılmadı (sadece küçük sentetik verilerle mantık doğrulaması yapıldı).
Bu nedenle **önce** Hücre 3'te `SMOKE_TEST = True` yapıp `Run All` çalıştırarak
birkaç dakikada tüm pipeline'ı (eğitim→saldırı→savunma→ablasyon→rapor) küçük
bir veri alt-kümesiyle uçtan uca doğrulamanız önerilir. Sorun çıkmazsa
`SMOKE_TEST = False` yapıp tekrar `Run All` ile tam ölçekli çalışmayı başlatın
(duman testi ve tam çalışma ayrı cache/checkpoint dosyaları kullanır, birbirini
bozmaz).

### 5. "Run All"
Yukarıdaki adımlar tamamlandıktan sonra `Runtime > Run all` çalıştırın. Notebook,
Colab oturumu koparsa bile (yeniden `Run All` ile) kaldığı yerden devam edecek
şekilde tasarlanmıştır (bkz. checkpoint mekanizması, Hücre 9 ve 14).

**Tahmini çalışma süresi (A100, 3 veri seti, 5 model, 7 tohum + savunma + ablasyon):**
Tablo/NetFlow verisi ile çalışıldığı için modeller küçüktür (milyon parametre altı);
tüm pipeline'ın A100'de birkaç saat içinde tamamlanması beklenir. Checkpoint mekanizması
sayesinde kesinti durumunda kaldığı yerden devam eder.


## 1. Çalışma Zamanı Doğrulama (Hard-GPU-Only)

In [ ]:
# ============================================================
# HÜCRE 1 — RUNTIME KONTROLÜ (HARD-GPU-ONLY MODU)
# Bu hücre, notebook'un geri kalanı çalışmadan ÖNCE GPU varlığını
# doğrular. GPU bulunamazsa notebook kasıtlı olarak DURDURULUR
# (CPU'da yanlışlıkla saatlerce/gunlerce calismayi engeller).
# ============================================================
import os, sys, subprocess, platform, datetime

RUN_ID = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%d_%H%M%S")
print(f"[{RUN_ID}] Calisma baslatildi (UTC).")
print(f"Python: {platform.python_version()} | Platform: {platform.platform()}")

try:
    import torch
except ImportError as e:
    raise RuntimeError(
        "PyTorch bulunamadi. Colab runtime'i bozuk olabilir. "
        "Runtime > Restart runtime deneyin."
    ) from e

CUDA_OK = torch.cuda.is_available()
if not CUDA_OK:
    # HARD-GPU-ONLY: GPU yoksa devam etmiyoruz.
    raise RuntimeError(
        "\n\n*** HARD-GPU-ONLY MOD: GPU BULUNAMADI ***\n"
        "Bu notebook sadece GPU calisma zamaninda calisacak sekilde tasarlandi.\n"
        "Colab menusunden: Runtime > Change runtime type > Hardware accelerator > GPU (A100)\n"
        "secip tekrar 'Run All' yapin.\n"
    )

gpu_name = torch.cuda.get_device_name(0)
cc_major, cc_minor = torch.cuda.get_device_capability(0)
total_mem_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
cuda_version = torch.version.cuda
torch_version = torch.__version__

print("=" * 60)
print("GPU DOGRULAMA RAPORU")
print("=" * 60)
print(f"GPU              : {gpu_name}")
print(f"Compute Capability: {cc_major}.{cc_minor}")
print(f"Toplam VRAM      : {total_mem_gb:.1f} GB")
print(f"PyTorch surumu   : {torch_version}")
print(f"CUDA surumu      : {cuda_version}")
print("=" * 60)

IS_A100 = "A100" in gpu_name
if not IS_A100:
    print(
        f"[UYARI] Beklenen GPU 'A100' idi, algilanan: '{gpu_name}'. "
        "Colab Pro+ kota/uygunluguna bagli olarak farkli bir GPU atanmis olabilir. "
        "Calismaya devam edilecek ancak bellek/hiz varsayimlari A100'e gore ayarlanmistir; "
        "daha kucuk bir GPU'da OOM (bellek yetersizligi) olusabilir."
    )

# Ampere (compute capability >= 8.0) ve sonrasinda bf16 autocast guvenle kullanilabilir.
USE_BF16 = cc_major >= 8
AMP_DTYPE = torch.bfloat16 if USE_BF16 else torch.float16
print(f"Otomatik Karisik Hassasiyet (AMP) veri tipi: {AMP_DTYPE}")

# Guvenlik/performans ayarlari (en guncel CUDA/PyTorch surumleriyle uyumlu, deterministik-yakin ayar)
torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

DEVICE = torch.device("cuda:0")
print("Hard-GPU-Only kontrolu BASARILI. Calisma GPU uzerinde devam edecek.")


## 2. Google Drive Bağlama ve Bağımlılık Kurulumu

In [ ]:
# ============================================================
# HÜCRE 2 — GOOGLE DRIVE BAGLAMA + UCRETSIZ/ACIK-KAYNAK PAKET KURULUMU
# Kullanilan tum kutuphaneler ucretsiz ve acik kaynaktir; ucretli/API-key
# gerektiren hicbir servis (OpenAI, vb.) kullanilmamaktadir.
# ============================================================
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import subprocess, sys, importlib

def pip_install(pkg_spec: str, no_deps: bool = False):
    """--force-reinstall --no-cache-dir kullanilir (bkz. asagidaki HATA DUZELTMESI
    notu): paketin TUM dosyalarinin (hem .py hem .so) AYNI, tutarli wheel'den
    gelmesini garanti eder. no_deps=True ise bagimliliklara DOKUNULMAZ (numpy/scipy
    gibi TEMEL paketleri YANLISLIKLA yukseltmemek icin -- bkz. asagidaki not)."""
    cmd = [sys.executable, "-m", "pip", "install", "-q", "--force-reinstall", "--no-cache-dir"]
    if no_deps:
        cmd.append("--no-deps")
    cmd.append(pkg_spec)
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        print(f"[UYARI] '{pkg_spec}' kurulumunda sorun olustu:\n{result.stderr[-800:]}")
    else:
        print(f"[OK{' (--no-deps)' if no_deps else ''}] {pkg_spec}")

# ============================================================
# HATA DUZELTMESI (v3.9): GERCEK KOK NEDEN bulundu -- pip'in --force-reinstall
# davranisi.
#
# ONEMLI PIP DETAYI: '--force-reinstall', '--no-deps' VERILMEDIGI SURECE,
# hedef paketin TUM BAGIMLILIKLARINI DA (numpy, scipy dahil) zorla en guncel
# surume yeniden kurar! v3.8'de numpy/scipy'yi REQUIRED_PACKAGES listesinden
# CIKARMISTIK ama scikit-learn/pandas HALA '--force-reinstall' (--no-deps
# OLMADAN) ile kuruluyordu -- yani numpy/scipy YINE DE, bu kez DOLAYLI olarak,
# PyPI'daki EN GUNCEL (ve o an bozuk olan, 2.5.0) surume zorlaniyordu. v3.8
# hicbir seyi cozmemis, ayni sorunu gizlice devam ettirmis.
#
# AYRICA numpy 2.5.0'in kendisinin (temmuz 2026 itibariyle COK YENI bir
# 'gecis' surumu) derlenmis uzantisinda gercek bir ic-tutarsizlik/paketleme
# sorunu oldugu goruluyor (topluluktaki bagimsiz raporlar: Open3D #7263,
# whisperX #1244, mapclassify #278 -- hepsi AYNI hatayi, Colab'da, bizim
# kodumuzla ILGISIZ projelerde bildirmis).
#
# DUZELTME (iki katmanli):
#   1) TUM paketler --no-deps ile kurulur -- pip'in HICBIR bagimliligi
#      (numpy/scipy dahil) yan etki olarak degistirmesi artik MUMKUN DEGIL.
#   2) numpy AYRICA, ACIKCA <2.5 ile SINIRLANIR (yani bozuk oldugu bilinen
#      2.5.0 'gecis' surumu degil, bir onceki KARARLI seri olan 2.4.x
#      kurulur) -- scipy ile AYNI cagirida, TEK pip cozumlemesiyle, boylece
#      ikisi birbiriyle uyumlu secilir.
# NOT: torch / torchvision Colab runtime'inda onceden kurulu gelir; surumu bozmamak
# icin torch'u bu listeye DAHIL ETMIYORUZ (bu, Hard-GPU-Only kararliligini bozabilir).
print("[DUZELTME] numpy (<2.5, bilinen-kararli seri) + scipy TEK cagride, --no-deps ile kuruluyor...")
_numpy_fix = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "--no-cache-dir", "numpy<2.5", "scipy"],
    capture_output=True, text=True
)
if _numpy_fix.returncode != 0:
    print(f"[UYARI] numpy/scipy kurulumunda sorun olustu:\n{_numpy_fix.stderr[-800:]}")
else:
    print("[OK] numpy<2.5 (kararli seri) + uyumlu scipy kuruldu.")

# HATA DUZELTMESI (v3.10): matplotlib 3.11.0 (12 Haziran 2026'da yayinlandi --
# COK yeni), font/metin isleme alt yapisini (ft2font, FreeType/HarfBuzz/SheenBidi)
# BASTAN YAZAN buyuk bir surum. Bu, tam da hata aldigimiz yer (ft2font.
# CharacterCodeType). numpy 2.5.0'da gordugumuzle AYNI desen: cok taze bir
# majör surum, muhtemelen Colab'in paketleme ortamiyla henuz tam olgunlasmamis
# bir uyumsuzluk. Onceki, uzun sureli test edilmis 3.10.x serisine (<3.11)
# SABITLENIYOR.
REQUIRED_PACKAGES = [
    "torchattacks>=3.5.1",   # FGSM/PGD ucretsiz, acik kaynak, saf PyTorch (CUDA derlemesi gerektirmez)
    "scikit-learn>=1.4",
    "pandas>=2.2",
    "matplotlib>=3.8,<3.11", # 3.11.0'in yeni font-altyapisi kaynakli sorunu ONLEMEK icin sabitlendi
    "seaborn>=0.13",
    "tabulate>=0.9",
    "openpyxl>=3.1",
]

# ONEMLI: HEPSI --no-deps ile kurulur -- boylece HICBIRI numpy/scipy'yi (veya
# birbirini) yan etki olarak degistiremez. Bu paketlerin kendi bagimliliklari
# (joblib, threadpoolctl, pytz, python-dateutil, pillow, fonttools, cycler vb.)
# zaten Colab'in fabrika imajinda kurulu geldigi icin --no-deps guvenlidir.
for pkg in REQUIRED_PACKAGES:
    pip_install(pkg, no_deps=True)

# Zorla yeniden kurulumdan SONRA, bellekte (sys.modules) Colab'in kernel baslarken
# onceden yuklemis olabilecegi ESKI matplotlib kalintisi varsa temizlenir (numpy/
# scipy'ye DOKUNULMADIGI icin bu islem onlar icin GEREKMEZ/yapilmiyor -- torch'un
# halihazirda baglandigi numpy ABI'siyle celiskiye girme riski yok).
_purged = [m for m in list(sys.modules) if m == "matplotlib" or m.startswith("matplotlib.") or m == "pylab"]
for _m in _purged:
    del sys.modules[_m]
if _purged:
    print(f"[TEMIZLIK] {len(_purged)} onbellekteki matplotlib modulu bellekten temizlendi "
          f"(taze/tutarli yeniden-yukleme icin) -> {sorted(_purged)[:3]}...")

# Hizli oz-test: kritik kutuphaneler bu hucrede SORUNSUZ ice aktarilabiliyor mu?
# Sorun varsa BURADA (Hucre 2'de), calismanin cok daha ilerisinde degil, hemen
# yakalanir.
print("\n[OZ-TEST] Kritik kutuphaneler test ediliyor...")
_self_test_failed = False
for _mod in ["numpy", "scipy", "sklearn", "pandas", "matplotlib"]:
    try:
        importlib.import_module(_mod)
        print(f"  [OK] {_mod}")
    except Exception as e:
        _self_test_failed = True
        print(f"  [HATA] {_mod}: {type(e).__name__}: {e}")
if _self_test_failed:
    print("\n[UYARI] Yukaridaki en az bir kutuphane ice aktarilamadi. Runtime'i "
          "yeniden baslatip ('Restart runtime') bu hucreyi tekrar calistirmayi deneyin.")
else:
    print("[OZ-TEST] Tum kritik kutuphaneler sorunsuz ice aktarildi.")

print("\nTum bagimliliklar kontrol edildi/kuruldu.")
print(
    "\nNOT (mimari tasarim karari): Bu calismada resmi 'mamba-ssm' CUDA-derlemeli "
    "paketi KASITLI olarak kullanilmamistir. mamba-ssm paketi ozel CUDA cekirdekleri "
    "icerdiginden, guncel/degisken CUDA-PyTorch surumlerinde (v2026+ Colab imajlari dahil) "
    "derleme hatalarina ve 'kesintisiz calisma' garantisinin bozulmasina yol acabilmektedir. "
    "Bunun yerine, Bolum 3.2'de aciklanan, Gu & Dao (2023) tarafindan tanimlanan secici "
    "durum-uzayi (S6) formulasyonunu birebir izleyen, SAF PyTorch ile yazilmis, donanimdan "
    "bagimsiz bir yeniden-uygulama kullanilmaktadir. Bu tercih, tablo/NetFlow verisinin "
    "kisa dizi uzunlugu (<=128) nedeniyle performans kaybi yaratmadan tam tekrarlanabilirlik "
    "saglamaktadir."
)


## 3. Genel Konfigürasyon ve Klasör Yapısı

In [ ]:
# ============================================================
# HÜCRE 3 — GENEL KONFIGURASYON, DRIVE KLASOR YAPISI, TEKRARLANABILIRLIK
# ============================================================
import os, random, json
import numpy as np
import torch

# ---------- 3.1 Drive klasor yapisi ----------
BASE_DIR = "/content/drive/MyDrive/BTD_Mamba_NIDS_Robustness"

PATHS = {
    "base":        BASE_DIR,
    "datasets":    f"{BASE_DIR}/datasets",        # HAM veri setleri BURAYA ONCEDEN indirilmis olmali
    "cache":       f"{BASE_DIR}/cache",           # on-islenmis tensor cache (.pt) - tekrar indirme/isleme onler
    "checkpoints": f"{BASE_DIR}/checkpoints",      # model agirliklari (resume icin)
    "figures":     f"{BASE_DIR}/figures",          # yayina-hazir grafikler (PNG 300dpi + PDF vektor)
    "tables":      f"{BASE_DIR}/tables",           # yayina-hazir tablolar (CSV + LaTeX + Markdown)
    "logs":        f"{BASE_DIR}/logs",             # calisma loglari, hata izleri
    "reports":     f"{BASE_DIR}/reports",          # ozet rapor (Markdown)
    "results":     f"{BASE_DIR}/results",          # ham sonuc DataFrame'leri (CSV/Parquet)
}
for p in PATHS.values():
    os.makedirs(p, exist_ok=True)
print("Drive klasor yapisi hazir:")
for k, v in PATHS.items():
    print(f"  {k:12s} -> {v}")

# ---------- 3.2 Beklenen ham veri seti dosyalari (ONCEDEN Drive'a indirilmis olmali) ----------
# Kullanici bu dosyalari BIR KEZ indirip 'datasets/' altina koymalidir. Kod, dosya zaten
# varsa TEKRAR INDIRMEZ (Hucre 5'teki get_or_fetch_dataset fonksiyonuna bakiniz).
DATASET_FILES = {
    "NSL-KDD": {
        "train": f"{PATHS['datasets']}/NSL-KDD/KDDTrain+.txt",
        "test":  f"{PATHS['datasets']}/NSL-KDD/KDDTest+.txt",
    },
    "NF-UNSW-NB15": {
        "full": f"{PATHS['datasets']}/NF-UNSW-NB15/NF-UNSW-NB15-v2.csv",
    },
    "CICIDS2018": {
        "full": f"{PATHS['datasets']}/CICIDS2018/CICIDS2018_sample.csv",
    },
}

# ---------- 3.3 Deney konfigurasyonu ----------
# ============================================================
# DUMAN TESTI (SMOKE TEST) MODU
# GEREKCE: Bu kod GERCEK Colab GPU'sunda ve GERCEK veri setleriyle hic
# calistirilmadi (sadece kucuk sentetik verilerle CPU'da mantik/entegrasyon
# testi yapildi). Gercek veride beklenmedik surprizler (bellek asimi, veri
# tipi hatalari, sinif dagilimi anomalileri) cikma ihtimaline karsi,
# SMOKE_TEST=True ile once KUCUK bir alt-kumeyle (birkaç dakikada biten),
# TUM pipeline asamalarini (egitim->saldiri->savunma->ablasyon->rapor) UCTAN
# UCA calistirip doğrulamaniz siddetle ONERILIR. Sorun yoksa SMOKE_TEST=False
# yapip 'Run All' ile tam olcekli calismayi baslatin.
# ============================================================
SMOKE_TEST = False  # <-- ILK CALISTIRMADA True YAPIN, sorun yoksa False'a cevirip tekrar calistirin

CONFIG = {
    "run_id": RUN_ID,
    "smoke_test": SMOKE_TEST,
    # Istatistiksel anlamlilik testleri (esli t-testi, Wilcoxon) icin n=3 YETERSIZDIR
    # (p-degerleri guvenilmez olur). n=7 ile hem makul bir hesaplama yuku hem de
    # kabul edilebilir istatistiksel guc (power) saglanir.
    "seeds": [13, 42, 123, 777, 2024, 2025, 31415] if not SMOKE_TEST else [13, 42],
    # NOT (revizyon): CICIDS2018 bir NEGATIF KONTROL olarak tutulur. Ham
    # ozellik sayisi (79) seq_len'i (64) astigi icin 15 ozellik KESILIR ve
    # hayatta kalan iki saldiri kategorisi (Bot, Infiltration) akis-tabanli
    # literaturde en az ayrilabilir olanlardandir. Bu korpusta DUSUK basarim
    # BEKLENIR; amaci, hicbir model ogrenmedigi durumda karsilastirma
    # protokolunun makul davranip davranmadigini sinamaktir.
    "datasets": ["NSL-KDD", "NF-UNSW-NB15", "CICIDS2018"],
    "models": ["Mamba", "Transformer", "CNN1D", "LSTM", "RandomForest"],
    "batch_size": 512,
    "epochs": 40 if not SMOKE_TEST else 3,
    "patience": 6,                            # erken durdurma sabri
    "learning_rate": 3e-4,
    "weight_decay": 1e-4,
    "seq_len": 64,                            # tablo ozellikleri pseudo-dizi olarak padlenir
    "d_model": 64,
    "n_layers": 3,
    # SMOKE_TEST=True iken her veri setinden sadece bu kadar ornek kullanilir
    # (egitim+test toplam) -- gercek veri boru hattini SANIYELER icinde test eder.
    "smoke_test_max_rows": 3000,
    "attacks": {
        "FGSM": {"eps_list": [0.01, 0.03, 0.05, 0.1]},
        "PGD":  {"eps_list": [0.01, 0.03, 0.05, 0.1], "alpha": 0.01, "steps": 10},
        "BlackBoxTransfer": {"eps_list": [0.03, 0.1]},
    },
    # ---- Adversarial training (savunma) degerlendirmesi icin ----
    "adversarial_training": {
        "enabled": True,
        "models": ["Mamba", "CNN1D"],   # en sagliksiz ve en sağlam adaylar karsilastirilir
        "eps": 0.05,                     # egitim sirasinda kullanilan pertürbasyon buyuklugu
        "mix_ratio": 0.5,                # her batch'in yuzde kaci cekismeli ornekle degistirilir
    },
    # ---- Hiperparametre adaleti ablasyonu icin ----
    "ablation": {
        "enabled": True,
        "d_model_values": [32, 64],      # her mimari IKI farkli kapasitede de test edilir
        "seeds": [13, 42],               # ablasyon icin daha az tohum (hesaplama yuku)
    },
    "email_notifications": True,
    # ONEMLI (v3 revizyonu): split stratejisi (Hucre 6) veya baska bir veri-etkileyen
    # degisiklik yapildiginda bu degeri bump edin -- cache (Hucre 6) ve checkpoint
    # (Hucre 9) dosya adlarina eklenir, boylece ESKI STRATEJIYLE uretilmis dosyalar
    # SESSIZCE yeniden kullanilmaz (train/test tutarsizligini onler).
    "data_version": "v3_groupsplit",
}

with open(f"{PATHS['logs']}/config_{RUN_ID}.json", "w") as f:
    json.dump(CONFIG, f, indent=2, ensure_ascii=False)
print("\nKonfigurasyon kaydedildi ->", f"{PATHS['logs']}/config_{RUN_ID}.json")

# ---------- 3.4 Tekrarlanabilirlik ----------
def seed_everything(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(CONFIG["seeds"][0])
print(f"Baslangic tohumu ayarlandi: {CONFIG['seeds'][0]}")


## 4. Otomatik E-posta Bildirim Sistemi

In [ ]:
# ============================================================
# HÜCRE 4 — OTOMATIK E-POSTA BILDIRIM SISTEMI (Gmail SMTP, UCRETSIZ)
# Gerekli Colab Secrets (sol panel -> anahtar simgesi):
#   GMAIL_USER          -> gonderen gmail adresi (orn. "sizinadresiniz@gmail.com")
#   GMAIL_APP_PASSWORD  -> Google Hesabi > Guvenlik > Uygulama Sifreleri'nden
#                          uretilen 16 haneli UYGULAMA SIFRESI (normal sifre DEGIL)
#   GMAIL_TO            -> bildirimlerin gonderilecegi adres (kendinize de olabilir)
# Bu secret'lar Colab notebook Secrets panelinde tanimlanmali ve bu notebook
# icin erisime izin verilmelidir. Hicbir ucretli API kullanilmamaktadir.
# ============================================================
import smtplib, ssl, traceback, functools, datetime, os
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText
from email.mime.base import MIMEBase
from email import encoders
from google.colab import userdata

def _get_secret(name: str):
    try:
        return userdata.get(name)
    except Exception:
        return None

GMAIL_USER = _get_secret("GMAIL_USER")
GMAIL_APP_PASSWORD = _get_secret("GMAIL_APP_PASSWORD")
GMAIL_TO = _get_secret("GMAIL_TO")

EMAIL_READY = all([GMAIL_USER, GMAIL_APP_PASSWORD, GMAIL_TO])
if not EMAIL_READY:
    print(
        "[UYARI] E-posta bildirimleri devre disi: GMAIL_USER / GMAIL_APP_PASSWORD / "
        "GMAIL_TO secret'larindan biri veya birkaci bulunamadi. Colab Secrets panelinden "
        "ekleyip bu notebook'a erisim izni verin. Deney yine de calisacaktir, sadece "
        "bildirim gonderilmeyecektir."
    )
else:
    print(f"[OK] E-posta bildirimleri aktif. Gonderen: {GMAIL_USER} | Alici: {GMAIL_TO}")

def send_email(subject: str, body: str, attachment_path: str = None):
    """Gmail SMTP (SSL, 465 portu) uzerinden ucretsiz bildirim e-postasi gonderir.
    Basarisiz olursa (agsizlik/limit vb.) sessizce loglar, ana akisi KESMEZ."""
    if not EMAIL_READY:
        return False
    try:
        msg = MIMEMultipart()
        msg["From"] = GMAIL_USER
        msg["To"] = GMAIL_TO
        msg["Subject"] = f"[BTD-NIDS-Robustness][{RUN_ID}] {subject}"
        timestamp = datetime.datetime.now(datetime.timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC")
        full_body = f"Zaman: {timestamp}\nRun ID: {RUN_ID}\n\n{body}"
        msg.attach(MIMEText(full_body, "plain", "utf-8"))

        if attachment_path and os.path.exists(attachment_path):
            with open(attachment_path, "rb") as f:
                part = MIMEBase("application", "octet-stream")
                part.set_payload(f.read())
            encoders.encode_base64(part)
            part.add_header(
                "Content-Disposition",
                f"attachment; filename={os.path.basename(attachment_path)}",
            )
            msg.attach(part)

        context = ssl.create_default_context()
        with smtplib.SMTP_SSL("smtp.gmail.com", 465, context=context, timeout=30) as server:
            server.login(GMAIL_USER, GMAIL_APP_PASSWORD)
            server.sendmail(GMAIL_USER, GMAIL_TO, msg.as_string())
        return True
    except Exception as e:
        print(f"[UYARI] E-posta gonderilemedi: {e}")
        return False

def notify(stage: str, message: str = "", attachment_path: str = None):
    """Onemli asama bildirimleri icin kisa yol."""
    ok = send_email(stage, message, attachment_path)
    tag = "gonderildi" if ok else "gonderilemedi/devre-disi"
    print(f"[BILDIRIM:{tag}] {stage} -> {message[:120]}")

class EmailOnError:
    """Kapsamli bir kod blogunu sarar; icinde ISTISNA olusursa TAM TRACEBACK'i
    e-posta ile gonderir ve ardindan istisnayi tekrar yukseltir (calismayi durdurur).
    Kullanim:
        with EmailOnError('Veri on-isleme asamasi'):
            ... kod ...
    """
    def __init__(self, stage_name: str):
        self.stage_name = stage_name

    def __enter__(self):
        notify(f"BASLADI: {self.stage_name}", f"'{self.stage_name}' asamasi basladi.")
        return self

    def __exit__(self, exc_type, exc_val, exc_tb):
        if exc_type is not None:
            tb_text = "".join(traceback.format_exception(exc_type, exc_val, exc_tb))
            log_path = f"{PATHS['logs']}/error_{self.stage_name.replace(' ', '_')}_{RUN_ID}.log"
            try:
                with open(log_path, "w") as f:
                    f.write(tb_text)
            except Exception:
                log_path = None
            notify(
                f"HATA: {self.stage_name}",
                f"'{self.stage_name}' asamasinda ISTISNA olustu ve calisma DURDU.\n\n"
                f"Hata: {exc_type.__name__}: {exc_val}\n\n"
                f"--- Tam traceback (son 3000 karakter) ---\n{tb_text[-3000:]}",
                attachment_path=log_path,
            )
            return False  # istisnayi yut-mama; yukari firlat, calisma dursun
        else:
            notify(f"TAMAMLANDI: {self.stage_name}", f"'{self.stage_name}' asamasi basariyla tamamlandi.")
        return False

# Ilk saglik-kontrolu bildirimi (secret'lar dogru mu diye test)
if EMAIL_READY:
    notify(
        "Calisma baslatildi",
        f"Notebook GPU={gpu_name if 'gpu_name' in globals() else 'bilinmiyor'} uzerinde baslatildi.\n"
        f"Konfigurasyon: {json.dumps(CONFIG, ensure_ascii=False)[:500]}...",
    )


## 4b. CICIDS2018 Ham Günlük Dosyalarının Otomatik Birleştirilmesi (opsiyonel)
Kaggle'dan (`solarmainframe/ids-intrusion-csv`, CSE-CIC-IDS2018'in yaygın CSV yansıması)
indirilen veri **10 ayrı günlük dosya** halinde gelir (örn. `02-14-2018.csv`, `03-01-2018.csv`, ...),
ancak geri kalan pipeline **tek bir dosya** (`CICIDS2018_sample.csv`) bekler. Bu hücre, kullanıcının
indirdiği günlük dosyaları `datasets/CICIDS2018/raw/` altına koyduğunu varsayar, bunları otomatik
tespit edip birleştirir. Hiç ham dosya bulunamazsa (ör. dosyalar zaten birleştirilmiş halde
`CICIDS2018_sample.csv` olarak konulduysa) bu hücre sessizce atlanır, sonraki adımı etkilemez.

In [ ]:
# ============================================================
# HÜCRE 4b — CICIDS2018 HAM GÜNLÜK CSV'LERİNİN OTOMATİK BİRLEŞTİRİLMESİ
#
# GEREKÇE: solarmainframe/ids-intrusion-csv (Kaggle) veri kümesi 10 ayrı
# günlük dosya olarak dağıtılır; notebook'un geri kalanı ise TEK bir
# CICIDS2018_sample.csv dosyası bekler (bkz. DATASET_FILES, Hücre 3).
# Bu hücre kullanıcının 'raw/' alt klasörüne koyduğu HERHANGİ SAYIDA günlük
# dosyayı otomatik tespit edip, aşağıdaki BİLİNEN veri kalitesi sorunlarına
# karşı koruma sağlayarak birleştirir:
#   1) SÜTUN TUTARSIZLIĞI: günler arasında sütun sayısı/adı farklılık
#      gösterebilir -> ORTAK sütun kesişimi kullanılır, hangi dosyanın
#      hangi sütunları dışarıda bıraktığı AÇIKÇA raporlanır.
#   2) TEKRARLANAN BAŞLIK SATIRLARI: CICFlowMeter çıktısında bazı günlerde
#      başlık satırının veri içinde TEKRAR ettiği bilinir (ör. 'Label'
#      sütununda literal "Label" değeri) -> bu satırlar tespit edilip atılır.
# Bu hücre IDEMPOTENT'tir: CICIDS2018_sample.csv zaten varsa (bir önceki
# çalıştırmadan kalmışsa) TEKRAR birleştirme yapmaz, doğrudan atlar.
# ============================================================
import os, glob
import pandas as pd

CICIDS2018_RAW_DIR = f"{PATHS['datasets']}/CICIDS2018/raw"
CICIDS2018_COMBINED_PATH = DATASET_FILES["CICIDS2018"]["full"]

def combine_cicids2018_raw_files(raw_dir: str, output_path: str, force: bool = False) -> bool:
    """Ham günlük CICIDS2018 CSV dosyalarını tek bir dosyada birleştirir.
    Basari/atlama durumunu bool olarak döner (calismayi DURDURMAZ)."""
    if os.path.exists(output_path) and not force:
        print(f"[ATLA] {output_path} zaten mevcut (force=True ile yeniden birlestirmeyi zorlayabilirsiniz).")
        return True

    raw_files = sorted(glob.glob(f"{raw_dir}/*.csv"))
    if not raw_files:
        print(f"[BILGI] '{raw_dir}' altinda ham CICIDS2018 CSV dosyasi bulunamadi -- "
              "bu adim atlaniyor (CICIDS2018_sample.csv zaten varsa sorun yok; yoksa "
              "Hucre 5, CICIDS2018'i EKSIK olarak raporlayacaktir).")
        return False

    print(f"[BIRLESTIRME] {len(raw_files)} ham dosya bulundu: {[os.path.basename(f) for f in raw_files]}")
    dfs, col_sets = [], {}
    for f in raw_files:
        try:
            d = pd.read_csv(f, low_memory=False)
        except Exception as e:
            print(f"  [UYARI] {os.path.basename(f)} okunamadi, atlaniyor: {e}")
            continue
        d.columns = [c.strip() for c in d.columns]
        col_sets[os.path.basename(f)] = set(d.columns)
        dfs.append((os.path.basename(f), d))

    if not dfs:
        print("[HATA] Hicbir ham dosya basariyla okunamadi.")
        return False

    # 1) Sutun tutarliligi: ortak kesisimi bul, farkliliklari ACIKCA raporla
    common_cols = set.intersection(*col_sets.values())
    for fname, cols in col_sets.items():
        extra = cols - common_cols
        if extra:
            preview = sorted(extra)[:5]
            print(f"  [NOT] {fname}: diger dosyalarda olmayan {len(extra)} sutun "
                  f"(ortak kesisimde KULLANILMAYACAK): {preview}{'...' if len(extra) > 5 else ''}")
    common_cols = sorted(common_cols)
    print(f"  [BILGI] {len(common_cols)} ORTAK sutun tum dosyalarda kullanilacak.")

    label_col = next((c for c in common_cols if c.lower() == "label"), None)
    cleaned, total_removed = [], 0
    for fname, d in dfs:
        d = d[common_cols]
        if label_col is not None:
            # 2) Tekrarlanan baslik satirlarini tespit et/at
            bad_mask = d[label_col].astype(str).str.strip().str.lower() == "label"
            n_bad = int(bad_mask.sum())
            if n_bad > 0:
                d = d[~bad_mask]
                total_removed += n_bad
        cleaned.append(d)

    combined = pd.concat(cleaned, ignore_index=True)
    if total_removed > 0:
        print(f"  [TEMIZLIK] {total_removed} tekrarlanan baslik satiri veriden cikarildi.")

    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    combined.to_csv(output_path, index=False)
    print(f"[TAMAMLANDI] {len(combined)} satir, {len(common_cols)} sutun -> {output_path}")
    return True

with EmailOnError("CICIDS2018 ham dosya birlestirme"):
    combine_cicids2018_raw_files(CICIDS2018_RAW_DIR, CICIDS2018_COMBINED_PATH)


## 5. Veri Seti Varlık Kontrolü

In [ ]:
# ============================================================
# HÜCRE 5 — VERI SETI VARLIK KONTROLU
# TASARIM KARARI: Veri setleri, kullanicinin talebi geregi calisma
# BASLAMADAN ONCE Drive'a indirilmis olmalidir (bkz. README / Hucre 3.2).
# Bu hucre veri setlerini INTERNETTEN TEKRAR TEKRAR INDIRMEZ; sadece
# varligini dogrular. Tek istisna: kucuk (~20MB) ve kaynagi kararli olan
# NSL-KDD icin, dosya Drive'da yoksa TEK SEFERLIK bir indirme denemesi
# yapilir ve sonuc kalicidir sekilde Drive'a yazilir (bir daha denenmez).
# ============================================================
import os, time, hashlib, urllib.request, urllib.error

NSL_KDD_MIRRORS = {
    "train": "https://raw.githubusercontent.com/jmnwong/NSL-KDD-Dataset/master/KDDTrain%2B.txt",
    "test":  "https://raw.githubusercontent.com/jmnwong/NSL-KDD-Dataset/master/KDDTest%2B.txt",
}

def _download_once(url: str, dest_path: str, timeout: int = 60) -> bool:
    """Tek seferlik, basit ve guvenli indirme. Basarisiz olursa False doner,
    TEKRAR DENEMEZ (kullanicinin 'defalarca uzun indirme olmasin' talebine uygun)."""
    os.makedirs(os.path.dirname(dest_path), exist_ok=True)
    tmp_path = dest_path + ".part"
    try:
        req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
        with urllib.request.urlopen(req, timeout=timeout) as resp, open(tmp_path, "wb") as out:
            out.write(resp.read())
        os.replace(tmp_path, dest_path)
        return True
    except Exception as e:
        print(f"[UYARI] Indirme basarisiz ({url}): {e}")
        if os.path.exists(tmp_path):
            os.remove(tmp_path)
        return False

def verify_dataset_availability(dataset_files: dict) -> dict:
    """Her veri seti icin dosya(lar)in Drive'da olup olmadigini kontrol eder.
    Eksikse (yalnizca NSL-KDD icin) tek seferlik otomatik indirme dener.
    Sonucta hala eksik olan veri setleri, calismadan haric tutulur ve
    kullaniciya net bir e-posta/konsol uyarisi ile bildirilir."""
    status = {}
    for name, files in dataset_files.items():
        missing = {k: v for k, v in files.items() if not os.path.exists(v)}
        if not missing:
            status[name] = "HAZIR (Drive'da mevcut)"
            continue

        if name == "NSL-KDD":
            print(f"[BILGI] {name} Drive'da bulunamadi, tek seferlik indirme deneniyor...")
            all_ok = True
            for split, dest_path in files.items():
                if split in missing:
                    ok = _download_once(NSL_KDD_MIRRORS.get(split, ""), dest_path)
                    all_ok = all_ok and ok
            status[name] = "HAZIR (otomatik indirildi)" if all_ok else "EKSIK (indirme basarisiz, manuel gerekli)"
        else:
            status[name] = "EKSIK (manuel indirme gerekli)"

    return status

with EmailOnError("Veri seti varlik kontrolu"):
    dataset_status = verify_dataset_availability(DATASET_FILES)

print("\n" + "=" * 60)
print("VERI SETI DURUM RAPORU")
print("=" * 60)
for name, st in dataset_status.items():
    print(f"  {name:16s} -> {st}")
print("=" * 60)

AVAILABLE_DATASETS = [d for d, st in dataset_status.items() if st.startswith("HAZIR")]
MISSING_DATASETS = [d for d, st in dataset_status.items() if not st.startswith("HAZIR")]

if MISSING_DATASETS:
    msg = (
        f"Su veri setleri Drive'da bulunamadi ve otomatik olarak temin edilemedi: "
        f"{MISSING_DATASETS}.\n\n"
        "Lutfen resmi kaynaklardan indirip README'de belirtilen klasor yapisina "
        "gore yerlestirin:\n"
        "  - NF-UNSW-NB15-v2: https://staff.itee.uq.edu.au/marius/NIDS_datasets/ "
        f"-> {PATHS['datasets']}/NF-UNSW-NB15/NF-UNSW-NB15-v2.csv\n"
        "  - CICIDS2018: https://www.unb.ca/cic/datasets/ids-2018.html "
        f"-> {PATHS['datasets']}/CICIDS2018/CICIDS2018_sample.csv\n\n"
        f"Calisma, MEVCUT olan veri setleriyle ({AVAILABLE_DATASETS}) devam edecektir."
    )
    print("\n[UYARI]", msg)
    notify("Eksik veri seti(leri)", msg)

if not AVAILABLE_DATASETS:
    raise RuntimeError(
        "HICBIR veri seti kullanima hazir degil. Lutfen en az bir veri setini "
        "(orn. NSL-KDD otomatik indirilmis olmali) dogrulayip tekrar calistirin."
    )

# Bu calismada kullanilacak veri setleri kumesini guncelle
CONFIG["datasets"] = [d for d in CONFIG["datasets"] if d in AVAILABLE_DATASETS]
print(f"\nBu calisma icin kullanilacak veri setleri: {CONFIG['datasets']}")


## 6. Veri Ön İşleme (ortak boru hattı, cache'lenir)

**Önemli:** İkili (binary) sınıflandırma hedefi olarak kullanılmaya devam eder,
ANCAK orijinal saldırı **alt-türü** (attack_category: DoS, Probe, R2L, U2R,
Exploits, Reconnaissance, vb.) de ayrıca saklanır. Bu, "hangi saldırı türü
hangi mimariyi en çok kandırıyor?" sorusunu Hücre 11/15'te yanıtlayabilmek içindir.

**Duman Testi (Smoke Test) Modu:** Hücre 3'teki `SMOKE_TEST` bayrağı `True`
yapılırsa, veri seti stratified olarak küçük bir alt-kümeye (`smoke_test_max_rows`,
varsayılan 3000 satır) indirgenir ve az epoch/tohumla çalışılır. **Bu kod daha
önce gerçek GPU'da/gerçek verilerle hiç çalıştırılmadığından, önce `SMOKE_TEST=True`
ile birkaç dakikalık bir uçtan-uca doğrulama yapmanız kuvvetle önerilir** —
sorun yoksa `SMOKE_TEST=False` yapıp tam ölçekli "Run All" çalıştırın. Duman
testi ve tam çalışma AYRI cache dosyaları kullanır, birbirini bozmaz.


In [ ]:
# ============================================================
# HÜCRE 6 — ORTAK ON-ISLEME BORU HATTI (tum veri setleri icin)
# Islenmis tensorler cache/ altina .pt olarak kaydedilir; ayni veri seti
# tekrar calistirildiginda ISLEME TEKRARLANMAZ, dogrudan cache'ten okunur.
#
# ONEMLI (v2): Ikili (binary: saldiri/normal) siniflandirma HEDEFI olarak
# kullanilmaya devam eder (modeller bu gorev icin egitilir), ANCAK orijinal
# SALDIRI ALT-TURU (attack_category: DoS, Probe, Exploits, Reconnaissance,
# vb.) da AYRICA saklanir. Bu, "hangi saldiri turu hangi mimarinin en cok
# kandirilmasina yol aciyor?" sorusunu Hucre 11/15'te YANITLAYABILMEK icindir
# -- salt ikili dogruluk bu soruyu gizler.
# ============================================================
import pandas as pd
import numpy as np
import torch
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split, GroupShuffleSplit

NSL_KDD_COLUMNS = [
    "duration","protocol_type","service","flag","src_bytes","dst_bytes","land",
    "wrong_fragment","urgent","hot","num_failed_logins","logged_in","num_compromised",
    "root_shell","su_attempted","num_root","num_file_creations","num_shells",
    "num_access_files","num_outbound_cmds","is_host_login","is_guest_login","count",
    "srv_count","serror_rate","srv_serror_rate","rerror_rate","srv_rerror_rate",
    "same_srv_rate","diff_srv_rate","srv_diff_host_rate","dst_host_count",
    "dst_host_srv_count","dst_host_same_srv_rate","dst_host_diff_srv_rate",
    "dst_host_same_src_port_rate","dst_host_srv_diff_host_rate","dst_host_serror_rate",
    "dst_host_srv_serror_rate","dst_host_rerror_rate","dst_host_srv_rerror_rate",
    "label","difficulty",
]

# NSL-KDD'nin ~22 spesifik saldiri adi, literaturdeki standart 4 ust-kategoriye
# esler (Tavallaee ve ark., 2009 ile tutarli). Esleme YOKSA "Other" ile isaretlenir
# (yeni/bilinmeyen bir saldiri adi cikarsa calisma KIRILMAZ).
NSL_KDD_CATEGORY_MAP = {
    "normal": "Benign",
    # DoS
    "back": "DoS", "land": "DoS", "neptune": "DoS", "pod": "DoS", "smurf": "DoS",
    "teardrop": "DoS", "apache2": "DoS", "udpstorm": "DoS", "processtable": "DoS",
    "worm": "DoS", "mailbomb": "DoS",
    # Probe
    "ipsweep": "Probe", "nmap": "Probe", "portsweep": "Probe", "satan": "Probe",
    "mscan": "Probe", "saint": "Probe",
    # R2L (Remote to Local)
    "ftp_write": "R2L", "guess_passwd": "R2L", "imap": "R2L", "multihop": "R2L",
    "phf": "R2L", "spy": "R2L", "warezclient": "R2L", "warezmaster": "R2L",
    "sendmail": "R2L", "named": "R2L", "snmpgetattack": "R2L", "snmpguess": "R2L",
    "xlock": "R2L", "xsnoop": "R2L", "httptunnel": "R2L",
    # U2R (User to Root)
    "buffer_overflow": "U2R", "loadmodule": "U2R", "perl": "U2R", "rootkit": "U2R",
    "ps": "U2R", "sqlattack": "U2R", "xterm": "U2R",
}

SPLIT_STRATEGY_LOG = {}  # veri seti -> kullanilan split stratejisinin aciklamasi (Hucre 16 final raporunda kullanilir)

def _group_aware_split(df: pd.DataFrame, dataset_name: str, group_cols=None, time_col: str = None,
                        test_size: float = 0.2, seed: int = 42) -> pd.Series:
    """Rastgele SATIR-bazli split yerine, mumkunse GRUP-farkindali (ayni akis/host-cifti
    hem train hem test'e DUSMEYECEK sekilde) veya ZAMAN-farkindali split uygular.

    GEREKCE (v3 metodolojik revizyonu -- hakem geri bildirimi): Rastgele satir-bazli
    split, ayni baglantinin/akisin (veya ayni kaynak-hedef IP ciftinin) hem egitim hem
    test setine dusmesine izin vererek MODEL DEGERLENDIRMESINI IYIMSER YONDE
    CARPITABILIR (bkz. Apruzzese ve ark., NIDS-ML degerlendirme kirilganlik literaturu).
    Bu fonksiyon, oncelik sirasiyla:
      1) GRUP-bazli split (ayni host-cifti/akis TAMAMEN train VEYA test'te kalir),
      2) ZAMAN-bazli (kronolojik) split (gec zaman dilimi = test),
      3) Son care olarak STRATIFIED RASTGELE split (eski davranis, ACIKCA loglanir/uyarilir)
    seklinde en iyi mevcut secenegi kullanir. Hangi stratejinin kullanildigi
    SPLIT_STRATEGY_LOG'a kaydedilir ve final raporda (Hucre 16) SEFFAF olarak belirtilir
    -- boylece hangi veri setinin hangi garantiye sahip oldugu makale metninde acikca
    ifade edilebilir."""
    n = len(df)

    # 1) Grup-bazli split (tercih edilen): host-cifti veya flow-ID mevcutsa
    usable_group_cols = [c for c in (group_cols or []) if c in df.columns]
    if usable_group_cols:
        if len(usable_group_cols) >= 2:
            group_key = df[usable_group_cols[0]].astype(str) + "__" + df[usable_group_cols[1]].astype(str)
        else:
            group_key = df[usable_group_cols[0]].astype(str)
        n_groups = group_key.nunique()
        if n_groups >= 10:  # anlamli bir gruplama icin en az birkac grup gerekir
            gss = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=seed)
            train_idx, test_idx = next(gss.split(df, groups=group_key))
            split = pd.Series("train", index=df.index)
            split.iloc[test_idx] = "test"
            msg = (f"GRUP-bazli split kullanildi (gruplama sutunlari: {usable_group_cols}, "
                   f"{n_groups} benzersiz grup). Ayni host-cifti/akis train VE test'e AYNI ANDA dusmez.")
            SPLIT_STRATEGY_LOG[dataset_name] = msg
            print(f"  [SPLIT] {dataset_name}: {msg}")
            return split
        else:
            print(f"  [SPLIT-UYARI] {dataset_name}: gruplama sutunlari bulundu ancak benzersiz grup "
                  f"sayisi cok dusuk ({n_groups}) -- grup-bazli split GUVENILIR olmayabilir, "
                  f"bir sonraki stratejiye (zaman-bazli) geciliyor.")

    # 2) Zaman-bazli (kronolojik) split: grup bilgisi yoksa/yetersizse ikinci en iyi secenek
    if time_col and time_col in df.columns:
        try:
            ts = pd.to_datetime(df[time_col], errors="coerce")
            if ts.notna().sum() / max(n, 1) > 0.9:  # zaman damgalarinin buyuk cogunlugu ayristirilabiliyor mu
                order = ts.sort_values(kind="mergesort").index
                cut = int(len(order) * (1 - test_size))
                split = pd.Series("train", index=df.index)
                split.loc[order[cut:]] = "test"
                msg = (f"ZAMAN-bazli (kronolojik) split kullanildi (sutun: {time_col}) -- grup/flow "
                       f"kimligi bulunamadigi icin ikinci en iyi secenek. Test seti EN GEC zaman "
                       f"dilimini icerir (gercekci 'gelecegi tahmin etme' senaryosu).")
                SPLIT_STRATEGY_LOG[dataset_name] = msg
                print(f"  [SPLIT] {dataset_name}: {msg}")
                return split
        except Exception as e:
            print(f"  [SPLIT-UYARI] {dataset_name}: zaman-bazli split denemesi basarisiz ({e}), "
                  f"bir sonraki stratejiye geciliyor.")

    # 3) Son care: stratified rastgele split (ONCEKI/eski davranis) -- ACIKCA uyarilir ve
    # loglanir, cunku bu strateji akis/host-duzeyinde SIZINTI (leakage) riski tasir.
    try:
        train_idx, test_idx = train_test_split(
            df.index, test_size=test_size, random_state=seed, stratify=df["binary_label"]
        )
    except (ValueError, KeyError):
        train_idx, test_idx = train_test_split(df.index, test_size=test_size, random_state=seed)
    split = pd.Series("train", index=df.index)
    split.loc[test_idx] = "test"
    msg = ("STRATIFIED RASTGELE split kullanildi (SATIR-bazli) -- bu veri setinde ne grup/flow "
           "kimligi (host-cifti, flow-ID) ne de guvenilir bir zaman damgasi bulunamadi. "
           "*** METODOLOJIK SINIRLAMA ***: bu strateji, ayni akisin/baglantinin hem train hem "
           "test setine dusmesine izin verebilir ve sonuclari IYIMSER YONDE ETKILEYEBILIR. "
           "Makalede bu ACIKCA bir sinirlama olarak belirtilmelidir.")
    SPLIT_STRATEGY_LOG[dataset_name] = msg
    print(f"  [SPLIT-UYARI] {dataset_name}: {msg}")
    return split


def _load_raw_dataframe(name: str) -> pd.DataFrame:
    if name == "NSL-KDD":
        train = pd.read_csv(DATASET_FILES[name]["train"], names=NSL_KDD_COLUMNS, header=None)
        test = pd.read_csv(DATASET_FILES[name]["test"], names=NSL_KDD_COLUMNS, header=None)
        train["split"], test["split"] = "train", "test"
        df = pd.concat([train, test], ignore_index=True)
        df = df.drop(columns=["difficulty"])
        df["binary_label"] = (df["label"] != "normal").astype(int)
        df["attack_category"] = df["label"].str.strip().str.lower().map(NSL_KDD_CATEGORY_MAP).fillna("Other")
        cat_cols = ["protocol_type", "service", "flag"]
    elif name == "NF-UNSW-NB15":
        df = pd.read_csv(DATASET_FILES[name]["full"], low_memory=False)
        df.columns = [c.strip() for c in df.columns]
        label_col = "Label" if "Label" in df.columns else "label"
        df["binary_label"] = df[label_col].astype(int)
        attack_col = "Attack" if "Attack" in df.columns else label_col
        df["attack_category"] = np.where(df["binary_label"] == 0, "Benign", df[attack_col].astype(str))
        df["label"] = df[attack_col].astype(str)
        # ONEMLI (v3 revizyonu): rastgele SATIR-bazli split yerine, ayni host-cifti
        # (IPV4_SRC_ADDR, IPV4_DST_ADDR) hem train hem test'e DUSMEYECEK sekilde
        # GRUP-bazli split kullanilir (akis-duzeyinde sizinti riskini azaltir).
        df["split"] = _group_aware_split(
            df, name, group_cols=["IPV4_SRC_ADDR", "IPV4_DST_ADDR"], time_col=None, test_size=0.2, seed=42
        )
        drop_cols = [c for c in ["IPV4_SRC_ADDR", "IPV4_DST_ADDR", label_col] if c in df.columns]
        df = df.drop(columns=drop_cols, errors="ignore")
        cat_cols = [c for c in df.select_dtypes(include=["object", "string"]).columns
                    if c not in ("label", "split", "attack_category")]
    elif name == "CICIDS2018":
        df = pd.read_csv(DATASET_FILES[name]["full"], low_memory=False)
        df.columns = [c.strip() for c in df.columns]
        label_col = "Label" if "Label" in df.columns else df.columns[-1]
        df["label"] = df[label_col].astype(str)
        df["binary_label"] = (df["label"].str.lower() != "benign").astype(int)
        df["attack_category"] = np.where(df["binary_label"] == 0, "Benign", df["label"])
        # ONEMLI (v3 revizyonu): CICIDS2018'in kamuya acik CSV surumlerinde IP/flow
        # kimligi sutunlari SIKLIKLA bulunmaz (gizlilik nedeniyle cikarilmis olabilir).
        # Bu nedenle olasi sutun adi varyantlari sirayla denenir; hicbiri yoksa zaman
        # damgasina (varsa), o da yoksa stratified rastgele split'e (ACIKCA uyarilarak)
        # geri donulur -- bkz. _group_aware_split(). Hangi stratejinin FIILEN kullanildigi
        # calisma ciktisinda ve final raporda (Hucre 16) SEFFAF olarak belirtilir.
        possible_group_cols = [
            ["Flow ID"], ["Src IP", "Dst IP"], ["Source IP", "Destination IP"],
            ["Src IP Addr", "Dst IP Addr"],
        ]
        chosen_group_cols = []
        for cand in possible_group_cols:
            if all(c in df.columns for c in cand):
                chosen_group_cols = cand
                break
        possible_time_cols = ["Timestamp", "timestamp", "Flow Start Time"]
        chosen_time_col = next((c for c in possible_time_cols if c in df.columns), None)
        df["split"] = _group_aware_split(
            df, name, group_cols=chosen_group_cols, time_col=chosen_time_col, test_size=0.2, seed=42
        )
        drop_id_cols = ["Flow ID", "Src IP", "Dst IP", "Source IP", "Destination IP",
                        "Src IP Addr", "Dst IP Addr"]
        df = df.drop(columns=[c for c in drop_id_cols if c in df.columns] + [label_col], errors="ignore")
        cat_cols = [c for c in df.select_dtypes(include=["object", "string"]).columns
                    if c not in ("label", "split", "attack_category")]
    else:
        raise ValueError(f"Bilinmeyen veri seti: {name}")

    df = df.replace([np.inf, -np.inf], np.nan).dropna()
    for c in cat_cols:
        df[c] = LabelEncoder().fit_transform(df[c].astype(str))
    return df

def load_and_preprocess(name: str, seq_len: int, smoke_test: bool = False, smoke_max_rows: int = 3000):
    """Veri setini yukler, olceklendirir, sabit uzunluklu (seq_len) sozde-dizi
    formatina donusturur (Mamba/Transformer/CNN/LSTM'in ortak girdisi icin) ve
    cache'ler. Doner: X_train, y_train, X_test, y_test, n_features,
    attack_category_train, attack_category_test (str numpy array).

    smoke_test=True ise, veri seti STRATIFIED olarak kucuk bir alt-kumeye
    (smoke_max_rows) indirgenir -- GERCEK veri boru hattini (kolon isimleri,
    tip donusumleri, olceklendirme, vb.) SANIYELER icinde uctan uca dogrulamak
    icindir. Duman testi ve tam-olcekli calisma AYRI cache dosyalari kullanir
    (birbirini SESSIZCE EZMEZ / KARISTIRMAZ)."""
    cache_suffix = "_smoke" if smoke_test else ""
    # ONEMLI (v3 revizyonu): cache dosya adina CONFIG['data_version'] eklenir. Boylece bu
    # REVIZYONDAN (grup-farkindali split) ONCEKI cache dosyalari YANLISLIKLA yeniden
    # kullanilmaz -- data_version degisince otomatik olarak yeniden islenir.
    cache_path = f"{PATHS['cache']}/{name}_seq{seq_len}_{CONFIG['data_version']}{cache_suffix}.pt"
    if os.path.exists(cache_path):
        print(f"[CACHE] {name} on-islenmis veri Drive cache'inden okunuyor: {cache_path}")
        d = torch.load(cache_path, weights_only=False)
        return (d["X_train"], d["y_train"], d["X_test"], d["y_test"], d["n_features"],
                d["attack_category_train"], d["attack_category_test"])

    print(f"[ISLENIYOR] {name} ham veriden on-isleniyor (bu islem bir kez yapilir)"
          f"{' [DUMAN TESTI MODU]' if smoke_test else ''}...")
    df = _load_raw_dataframe(name)

    if smoke_test and len(df) > smoke_max_rows:
        try:
            df, _ = train_test_split(
                df, train_size=smoke_max_rows, stratify=df["binary_label"], random_state=42
            )
        except ValueError:
            # Stratify basarisiz olursa (orn. bir sinifta cok az ornek) rastgele orneklemeye don
            df = df.sample(n=smoke_max_rows, random_state=42)
        print(f"[DUMAN TESTI] {name} -> {len(df)} satira indirgendi (stratified).")

    feature_cols = [c for c in df.columns if c not in ("label", "binary_label", "split", "attack_category")]

    train_df = df[df["split"] == "train"]
    test_df = df[df["split"] == "test"]
    if len(test_df) == 0:  # split kolonu yoksa stratified split uygula
        train_df, test_df = train_test_split(
            df, test_size=0.2, random_state=42, stratify=df["binary_label"]
        )

    scaler = StandardScaler()
    X_train_raw = scaler.fit_transform(train_df[feature_cols].values.astype(np.float32))
    X_test_raw = scaler.transform(test_df[feature_cols].values.astype(np.float32))
    y_train = train_df["binary_label"].values.astype(np.int64)
    y_test = test_df["binary_label"].values.astype(np.int64)
    cat_train = train_df["attack_category"].values.astype(str)
    cat_test = test_df["attack_category"].values.astype(str)

    n_features = X_train_raw.shape[1]

    def to_pseudo_sequence(X: np.ndarray) -> np.ndarray:
        # Her ornegi (seq_len, 1) sekline getirir; ozellik sayisi seq_len'den azsa
        # sifir ile doldurulur (pad), fazlaysa kirpilir (truncate). Bu, tum
        # mimarilerin (Mamba/Transformer/CNN/LSTM) AYNI girdi formatini
        # kullanmasini saglayarak ADIL bir karsilastirma sunar.
        n = X.shape[1]
        if n >= seq_len:
            X = X[:, :seq_len]
        else:
            pad = np.zeros((X.shape[0], seq_len - n), dtype=np.float32)
            X = np.concatenate([X, pad], axis=1)
        return X[:, :, None]  # (N, seq_len, 1)

    X_train = torch.tensor(to_pseudo_sequence(X_train_raw), dtype=torch.float32)
    X_test = torch.tensor(to_pseudo_sequence(X_test_raw), dtype=torch.float32)
    y_train_t = torch.tensor(y_train, dtype=torch.long)
    y_test_t = torch.tensor(y_test, dtype=torch.long)

    torch.save(
        {"X_train": X_train, "y_train": y_train_t, "X_test": X_test, "y_test": y_test_t,
         "n_features": n_features, "attack_category_train": cat_train, "attack_category_test": cat_test},
        cache_path,
    )
    print(f"[CACHE-YAZILDI] {name} -> {cache_path} "
          f"(train={len(X_train)}, test={len(X_test)}, orijinal_ozellik_sayisi={n_features}, "
          f"saldiri_alt-turu_sayisi={len(set(cat_test)) - 1})")
    return X_train, y_train_t, X_test, y_test_t, n_features, cat_train, cat_test

# Tum kullanilabilir veri setlerini on-isle / cache'ten yukle
DATA_CACHE = {}
with EmailOnError("Veri on-isleme (tum veri setleri)"):
    for ds_name in CONFIG["datasets"]:
        X_tr, y_tr, X_te, y_te, n_feat, cat_tr, cat_te = load_and_preprocess(
            ds_name, CONFIG["seq_len"],
            smoke_test=CONFIG.get("smoke_test", False),
            smoke_max_rows=CONFIG.get("smoke_test_max_rows", 3000),
        )
        DATA_CACHE[ds_name] = {
            "X_train": X_tr, "y_train": y_tr, "X_test": X_te, "y_test": y_te,
            "n_features": n_feat, "attack_category_train": cat_tr, "attack_category_test": cat_te,
        }
        n_categories = len(set(cat_te.tolist()) - {"Benign"})
        print(f"  {ds_name}: X_train={tuple(X_tr.shape)}, X_test={tuple(X_te.shape)}, "
              f"pozitif_oran(train)={y_tr.float().mean():.4f}, saldiri_alt-turu_sayisi={n_categories} "
              f"({sorted(set(cat_te.tolist()) - {'Benign'})})")

print("\n" + "=" * 60)
print("SPLIT STRATEJISI OZETI (seffaflik icin -- final raporda da yer alir)")
print("=" * 60)
for ds_name in CONFIG["datasets"]:
    if ds_name == "NSL-KDD":
        SPLIT_STRATEGY_LOG.setdefault(
            ds_name, "Resmi onceden-bolunmus KDDTrain+/KDDTest+ dosyalari kullanildi; "
                     "turetilmis bolme yok, bolme kaynakli sizinti kanali yok.")
    print(f"  [{ds_name}] {SPLIT_STRATEGY_LOG.get(ds_name, '(KAYIT YOK -- cache''ten yuklendi)')}")
print("=" * 60)

# ============================================================
# REVIZYON (kanitlanabilirlik): Cache'ten yuklenen veri setleri icin
# SPLIT_STRATEGY_LOG BOS kalir; bu durumda bolme garantisi IDDIA edilmis
# ama KANITLANMAMIS olur. Asagidaki blok, eksik kayitlari ham veriden
# YENIDEN TURETEREK doldurur -- bolme deterministik oldugu icin
# (GroupShuffleSplit, random_state=42) yeniden egitim GEREKMEZ.
# ============================================================
eksik = [d for d in CONFIG["datasets"] if d not in SPLIT_STRATEGY_LOG]
if eksik:
    print(f"\n[REVIZYON] {eksik} icin bolme kaydi yok (cache'ten yuklendi).")
    print("           Ham veriden yeniden turetilip DOGRULANIYOR...")
    for ds_name in eksik:
        try:
            _df = _load_raw_dataframe(ds_name)   # split sutununu yeniden uretir
            if "split" in _df.columns:
                _n_tr = int((_df["split"] == "train").sum())
                _n_te = int((_df["split"] == "test").sum())
                print(f"  [{ds_name}] yeniden turetildi: train={_n_tr:,} test={_n_te:,}")
                print(f"             strateji: {SPLIT_STRATEGY_LOG.get(ds_name, '(bilinmiyor)')}")
            else:
                print(f"  [{ds_name}] resmi bolme dosyalari kullanildi.")
            del _df
        except Exception as _e:
            print(f"  [{ds_name}] yeniden turetme basarisiz: {type(_e).__name__}: {_e}")
    with open(f"{PATHS['logs']}/split_strategy_{RUN_ID}.json", "w") as f:
        json.dump(SPLIT_STRATEGY_LOG, f, indent=2, ensure_ascii=False)
    print(f"\n[KAYDEDILDI] {PATHS['logs']}/split_strategy_{RUN_ID}.json")


## 6b. Ön İşleme Sonrası Otomatik Veri Sağlık Kontrolü

Gerçek veride NaN/Inf artıkları, aşırı sınıf dengesizliği, sıfır-varyanslı
özellikler veya boş saldırı alt-kategorileri gibi "ilk çalıştırmada ortaya
çıkan" sorunları, saatlerce sürecek asıl eğitime başlamadan ÖNCE tespit eder
ve bilgilendirir (hata fırlatmaz, sadece uyarır).


In [ ]:
# ============================================================
# HÜCRE 6b — ON-ISLEME SONRASI OTOMATIK VERI SAGLIK KONTROLU
#
# GEREKCE: Bu kod daha once GERCEK NF-UNSW-NB15/CICIDS2018 verisiyle
# calistirilmadi (sadece sentetik veriyle test edildi). Gercek veride
# NaN/Inf artiklari, asiri sinif dengesizligi, sifir-varyansli ozellikler
# veya bos saldiri alt-kategorileri gibi 'ilk calistirmada ortaya cikan'
# sorunlar beklenebilir. Bu hucre, pipeline SAATLERCE surecek asil egitime
# baslamadan ONCE bu sorunlari TESPIT EDER, konsola ve e-postaya BILGILENDIRICI
# bir tanilama raporu yazar (hata firlatmaz, sadece uyarir) -- boylece
# kullanici veriyle ilgili bir sorunu erken fark edebilir.
# ============================================================
def validate_preprocessed_data(data_cache: dict) -> dict:
    report_lines = ["VERI SAGLIK KONTROLU RAPORU", "=" * 50]
    warnings_found = []

    for ds_name, d in data_cache.items():
        report_lines.append(f"\n[{ds_name}]")

        # 1) NaN / Inf kontrolu (guvenlik agi -- normalde on-isleme bunu onler)
        for split in ["X_train", "X_test"]:
            X = d[split]
            n_nan = torch.isnan(X).sum().item()
            n_inf = torch.isinf(X).sum().item()
            if n_nan > 0 or n_inf > 0:
                msg = f"  [UYARI] {split}: {n_nan} NaN, {n_inf} Inf deger bulundu!"
                report_lines.append(msg)
                warnings_found.append(f"{ds_name}/{split}: NaN/Inf tespit edildi")
            else:
                report_lines.append(f"  {split}: NaN/Inf yok (temiz).")

        # 2) Sinif dengesizligi raporu (egitim ve test)
        for split_name, y in [("train", d["y_train"]), ("test", d["y_test"])]:
            counts = torch.bincount(y)
            total = counts.sum().item()
            ratios = (counts.float() / total).tolist()
            report_lines.append(f"  y_{split_name} sinif dagilimi: {counts.tolist()} "
                                 f"(oranlar: {[f'{r:.4f}' for r in ratios]})")
            min_ratio = min(ratios) if ratios else 0
            if min_ratio < 0.01:
                msg = (f"  [UYARI] {ds_name}/y_{split_name}: azinlik sinifi oldukca kucuk "
                       f"(%{min_ratio*100:.2f}) -- agirlikli loss/dengelenmis-dogruluk kullanildigindan "
                       f"(bkz. Hucre 9) egitim yine de calisir, ancak istatistiksel guc dusuk olabilir.")
                report_lines.append(msg)
                warnings_found.append(f"{ds_name}/y_{split_name}: asiri sinif dengesizligi (%{min_ratio*100:.2f})")
            if len(counts) < 2:
                msg = f"  [KRITIK UYARI] {ds_name}/y_{split_name}: SADECE TEK SINIF mevcut! AUC/F1 hesaplanamaz."
                report_lines.append(msg)
                warnings_found.append(f"{ds_name}/y_{split_name}: tek sinif (KRITIK)")

        # 3) Sifir-varyansli ozellik kontrolu (StandardScaler sonrasi)
        std_per_feature = d["X_train"].std(dim=0).squeeze()
        n_zero_var = (std_per_feature < 1e-6).sum().item()
        if n_zero_var > 0:
            msg = (f"  [UYARI] {n_zero_var} ozellik sifira-yakin varyansa sahip "
                   f"(egitim setinde sabit deger) -- bu ozellikler modele bilgi katkisi saglamaz.")
            report_lines.append(msg)
            warnings_found.append(f"{ds_name}: {n_zero_var} sifir-varyansli ozellik")

        # 4) Saldiri alt-kategorisi yeterlilik kontrolu (Hucre 11/15 icin)
        cat_test = d.get("attack_category_test")
        if cat_test is not None:
            from collections import Counter
            cat_counts = Counter(cat_test.tolist())
            cat_counts.pop("Benign", None)
            usable_cats = {c: n for c, n in cat_counts.items() if n >= 5}
            report_lines.append(f"  Saldiri alt-kategorileri (test, n>=5 -> analiz edilebilir): "
                                 f"{usable_cats if usable_cats else '(YOK)'}")
            if not usable_cats:
                msg = (f"  [UYARI] {ds_name}: yeterli orneğe sahip (n>=5) saldiri alt-kategorisi YOK -- "
                       f"T3_saldiri_turu_kirilganligi tablosu bu veri seti icin BOS kalabilir.")
                report_lines.append(msg)
                warnings_found.append(f"{ds_name}: kullanilabilir saldiri alt-kategorisi yok")

    report_text = "\n".join(report_lines)
    print(report_text)

    log_path = f"{PATHS['logs']}/data_validation_{RUN_ID}.log"
    with open(log_path, "w") as f:
        f.write(report_text)

    if warnings_found:
        notify(
            "Veri Sagligi Kontrolu: uyarilar bulundu",
            f"{len(warnings_found)} uyari tespit edildi (calisma DURMADI, sadece bilgilendirme):\n\n"
            + "\n".join(f"- {w}" for w in warnings_found)
            + f"\n\nTam rapor: {log_path}",
            attachment_path=log_path,
        )
    else:
        print("\n[OK] Veri saglik kontrolunde herhangi bir sorun tespit edilmedi.")

    return {"warnings": warnings_found, "report_path": log_path}

_validation_result = validate_preprocessed_data(DATA_CACHE)


### 6c. Tek-Eşik Ayrılabilirlik Denetimi (REVİZYON)

Bir korpusta **tek bir özelliğin tek bir eşikle** sınıfları ayırması, o
kıyaslamanın mimarileri ayırt edemeyeceğinin işaretidir. Bu hücre her veri seti
için bunu ölçer.

**Kritik nokta:** dengesiz bir sette tek-eşik doğruluğu çoğunluk sınıfı taban
çizgisinin altına inemez. Bu nedenle taban çizgisi de raporlanır ve bir özellik
ancak tabanı belirgin biçimde aşarsa aday sayılır.


In [ ]:
# ============================================================
# HÜCRE 6c — TEK-ESIK AYRILABILIRLIK DENETIMI (REVIZYON)
# ============================================================
def _en_iyi_tek_esik(x, y):
    """Tek bir esikle ulasilabilen EN YUKSEK dogruluk -- TAM cozum.

    Kaba kuvvet (N esik x N ornek) yerine siralamaya dayali O(N log N)
    hesap kullanilir. Bu yalnizca daha hizli degil, ayni zamanda daha
    KESINDIR: sabit sayida nicelik esigini taramak yerine MUMKUN TUM
    esikleri kapsar. Her iki kutup da (x>t -> 1 ve x>t -> 0) denenir;
    dejenere "hepsine ayni sinifi ver" kurallari da dahil edilir.
    """
    y = np.asarray(y).astype(np.int64)
    N = len(y)
    if N == 0:
        return 0.0
    o = np.argsort(x, kind="mergesort")
    ys = y[o]
    c1 = np.cumsum(ys)                    # ilk k ornekteki 1 sayisi
    c0 = np.arange(1, N + 1) - c1         # ilk k ornekteki 0 sayisi
    P = int(ys.sum()); Ng = N - P
    dogru      = c0 + (P - c1)            # kural: x > t  -> 1
    dogru_ters = c1 + (Ng - c0)           # kural: x > t  -> 0
    en = max(int(dogru.max()), int(dogru_ters.max()), P, Ng)
    return float(en) / N

def tek_esik_denetimi(data_cache, top_k=10):
    """Her veri seti icin tek-esikle ulasilabilen en yuksek dogruluk."""
    rapor = {}
    for ds, d in data_cache.items():
        X = d["X_train"].squeeze(-1).numpy()      # (N, seq_len)
        y = d["y_train"].numpy()
        n_feat = d["n_features"]
        taban = float(max(y.mean(), 1 - y.mean()))
        # yalnizca GERCEK ozellikler (dolgu pozisyonlari haric)
        idx = list(range(min(n_feat, X.shape[1])))
        skor = {i: _en_iyi_tek_esik(X[:, i], y) for i in idx}
        sirali = sorted(skor.items(), key=lambda kv: -kv[1])[:top_k]
        rapor[ds] = {"taban_cizgisi": taban,
                     "en_yuksek": [(int(i), v) for i, v in sirali]}
        print(f"\n[{ds}]  cogunluk-sinifi taban cizgisi = {taban:.4f}")
        for i, v in sirali:
            bayrak = "  <-- TABANI BELIRGIN ASIYOR" if v - taban > 0.20 else ""
            print(f"   ozellik #{i:3d}  tek-esik dogruluk = {v:.4f}"
                  f"  (taban+{v - taban:+.4f}){bayrak}")
    return rapor

SEPARABILITY = tek_esik_denetimi(DATA_CACHE)
with open(f"{PATHS['logs']}/separability_{RUN_ID}.json", "w") as f:
    json.dump(SEPARABILITY, f, indent=2)
print(f"\n[KAYDEDILDI] {PATHS['logs']}/separability_{RUN_ID}.json")
print("\nNOT: ozellik indeksleri, on-islemede kullanilan feature_cols SIRASINA")
print("karsilik gelir; ad eslemesi icin Hucre 6'daki _load_raw_dataframe cikti")
print("sutun sirasina bakiniz.")


## 7. Model Mimarileri

Bu bölümde dört mimari tanımlanır: **Mamba** (Selective State Space Model / S6,
Gu & Dao 2023'ün formülasyonunu izleyen, saf PyTorch ile yeniden uygulanmış hali),
**Transformer**, **CNN1D** ve **LSTM**. Tüm mimariler aynı girdi formatını
`(batch, seq_len, 1)` kullanır — bu, aralarındaki karşılaştırmayı **adil** kılar.


In [ ]:
# ============================================================
# HÜCRE 7 — MODEL MIMARILERI
# Tum modeller AYNI girdi formatini (B, L, 1) kullanir -> adil karsilastirma.
# ============================================================
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

# ---------------------------------------------------------------
# 7.1 SECICI DURUM-UZAYI (S6 / "Mamba") BLOGU — SAF PyTorch YENIDEN-UYGULAMA
# Gu & Dao (2023), "Mamba: Linear-Time Sequence Modeling with Selective
# State Spaces" makalesindeki secici tarama (selective scan) formulasyonunu
# izler: h_t = Abar_t * h_{t-1} + Bbar_t * x_t ;  y_t = C_t * h_t + D * x_t
# Ozel CUDA cekirdegi GEREKTIRMEZ (bkz. Hucre 2 mimari tasarim notu).
# ---------------------------------------------------------------
class SelectiveSSMBlock(nn.Module):
    def __init__(self, d_model: int, d_state: int = 16, expand: int = 2):
        super().__init__()
        self.d_model = d_model
        self.d_inner = expand * d_model
        self.d_state = d_state

        self.in_proj = nn.Linear(d_model, 2 * self.d_inner)
        self.conv1d = nn.Conv1d(
            self.d_inner, self.d_inner, kernel_size=4, padding=3, groups=self.d_inner
        )
        # Girdiye bagli (selective) delta, B, C projeksiyonlari
        self.x_proj = nn.Linear(self.d_inner, self.d_state * 2 + self.d_inner)
        self.dt_proj = nn.Linear(self.d_inner, self.d_inner)

        A = torch.arange(1, self.d_state + 1, dtype=torch.float32).repeat(self.d_inner, 1)
        self.A_log = nn.Parameter(torch.log(A))          # (d_inner, d_state)
        self.D = nn.Parameter(torch.ones(self.d_inner))  # atlama (skip) baglantisi
        self.out_proj = nn.Linear(self.d_inner, d_model)
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (B, L, d_model)
        B, L, _ = x.shape
        residual = x
        xz = self.in_proj(x)                                    # (B, L, 2*d_inner)
        x_in, z = xz.chunk(2, dim=-1)                            # her biri (B, L, d_inner)

        x_conv = self.conv1d(x_in.transpose(1, 2))[:, :, :L]     # nedensel (causal) konvolusyon
        x_conv = F.silu(x_conv.transpose(1, 2))                  # (B, L, d_inner)

        dbc = self.x_proj(x_conv)                                 # (B, L, d_state*2 + d_inner)
        delta, Bm, Cm = torch.split(dbc, [self.d_inner, self.d_state, self.d_state], dim=-1)
        delta = F.softplus(self.dt_proj(delta))                  # (B, L, d_inner) > 0

        A = -torch.exp(self.A_log)                               # (d_inner, d_state), kararlilik icin negatif

        # Ayrıklaştırma (discretization) - zero-order hold
        # delta: (B,L,d_inner) -> (B,L,d_inner,1) ; A: (d_inner,d_state)
        dA = torch.exp(delta.unsqueeze(-1) * A)                  # (B, L, d_inner, d_state)
        dB = delta.unsqueeze(-1) * Bm.unsqueeze(2)                # (B, L, d_inner, d_state)

        h = torch.zeros(B, self.d_inner, self.d_state, device=x.device, dtype=x.dtype)
        ys = []
        # Dizi uzunlugu tablo/NetFlow verisinde kisadir (<=128), bu nedenle
        # zaman-adimi dongusu GPU'da performans darbogazi YARATMAZ.
        for t in range(L):
            h = dA[:, t] * h + dB[:, t] * x_conv[:, t].unsqueeze(-1)
            y_t = torch.einsum("bds,bs->bd", h, Cm[:, t])
            ys.append(y_t)
        y = torch.stack(ys, dim=1)                                # (B, L, d_inner)
        y = y + x_conv * self.D
        y = y * F.silu(z)
        out = self.out_proj(y)
        return self.norm(out + residual)


class MambaNIDS(nn.Module):
    """Mamba-tabanli (secici durum-uzayi) siniflandirici."""
    def __init__(self, seq_len: int, d_model: int = 64, n_layers: int = 3,
                 d_state: int = 16, n_classes: int = 2):
        super().__init__()
        self.embed = nn.Linear(1, d_model)
        self.pos_embed = nn.Parameter(torch.zeros(1, seq_len, d_model))
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        self.blocks = nn.ModuleList([SelectiveSSMBlock(d_model, d_state) for _ in range(n_layers)])
        self.head = nn.Sequential(
            nn.LayerNorm(d_model), nn.Linear(d_model, d_model // 2), nn.GELU(),
            nn.Linear(d_model // 2, n_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (B, L, 1)
        h = self.embed(x) + self.pos_embed
        for blk in self.blocks:
            h = blk(h)
        pooled = h.mean(dim=1)
        return self.head(pooled)


# ---------------------------------------------------------------
# 7.2 BASLANGIC (BASELINE) MIMARILER
# ---------------------------------------------------------------
class TransformerNIDS(nn.Module):
    def __init__(self, seq_len: int, d_model: int = 64, n_layers: int = 3,
                 n_heads: int = 4, n_classes: int = 2):
        super().__init__()
        self.embed = nn.Linear(1, d_model)
        self.pos_embed = nn.Parameter(torch.zeros(1, seq_len, d_model))
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=d_model * 4,
            dropout=0.1, batch_first=True, activation="gelu",
        )
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=n_layers)
        self.head = nn.Sequential(
            nn.LayerNorm(d_model), nn.Linear(d_model, d_model // 2), nn.GELU(),
            nn.Linear(d_model // 2, n_classes),
        )

    def forward(self, x):
        h = self.embed(x) + self.pos_embed
        h = self.encoder(h)
        return self.head(h.mean(dim=1))


class CNN1DNIDS(nn.Module):
    def __init__(self, seq_len: int, d_model: int = 64, n_classes: int = 2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(1, d_model, kernel_size=5, padding=2), nn.BatchNorm1d(d_model), nn.ReLU(),
            nn.Conv1d(d_model, d_model, kernel_size=5, padding=2), nn.BatchNorm1d(d_model), nn.ReLU(),
            nn.AdaptiveAvgPool1d(1),
        )
        self.head = nn.Sequential(nn.Linear(d_model, d_model // 2), nn.GELU(), nn.Linear(d_model // 2, n_classes))

    def forward(self, x):
        h = self.net(x.transpose(1, 2)).squeeze(-1)
        return self.head(h)


class LSTMNIDS(nn.Module):
    def __init__(self, seq_len: int, d_model: int = 64, n_layers: int = 2, n_classes: int = 2):
        super().__init__()
        self.lstm = nn.LSTM(1, d_model, num_layers=n_layers, batch_first=True, bidirectional=True)
        self.head = nn.Sequential(
            nn.LayerNorm(d_model * 2), nn.Linear(d_model * 2, d_model), nn.GELU(),
            nn.Linear(d_model, n_classes),
        )

    def forward(self, x):
        out, _ = self.lstm(x)
        pooled = out.mean(dim=1)
        return self.head(pooled)


def build_model(name: str, seq_len: int, d_model: int, n_layers: int, n_classes: int = 2) -> nn.Module:
    if name == "Mamba":
        return MambaNIDS(seq_len, d_model, n_layers, n_classes=n_classes)
    elif name == "Transformer":
        return TransformerNIDS(seq_len, d_model, n_layers, n_classes=n_classes)
    elif name == "CNN1D":
        return CNN1DNIDS(seq_len, d_model, n_classes=n_classes)
    elif name == "LSTM":
        return LSTMNIDS(seq_len, d_model, n_layers=max(1, n_layers - 1), n_classes=n_classes)
    else:
        raise ValueError(f"Bilinmeyen model: {name}")

print("Model mimarileri tanimlandi: Mamba(S6), Transformer, CNN1D, LSTM.")


### 7c. S6 Uygulamasının Doğrulanması (REVİZYON)

Çalışmanın ana bulgusu, **yeniden uygulanan tek bileşen** hakkındadır. Bu nedenle
uygulamanın doğruluğu varsayılmaz, ölçülür. Üç bağımsız kontrol:

1. **CHECK 1** — resmî `selective_scan_ref` ile karşılaştırma. Bu fonksiyon saf
   PyTorch'tur; CUDA derlemesi gerektirmez, kaynağı doğrudan ayıklanabilir.
2. **CHECK 2** — özyinelemenin kapalı-form çözümüyle karşılaştırma; durum
   taşımadan, $O(L^2)$ doğrudan toplamla hesaplanır (döngüden bağımsız).
3. **CHECK 3** — sonlu farklarla gradyan kontrolü; bozuk bir geri geçiş her
   FGSM/PGD sonucunu bozardı.


In [ ]:
# ============================================================
# HÜCRE 7c — S6 UYGULAMA DOGRULAMASI (REVIZYON)
# ============================================================
import ast as _ast, urllib.request

def _referans_scan():
    """Resmi selective_scan_ref'i temin et (paket veya kaynak ayiklama)."""
    try:
        from mamba_ssm.ops.selective_scan_interface import selective_scan_ref
        return selective_scan_ref, "kurulu mamba-ssm"
    except Exception:
        pass
    try:
        url = ("https://raw.githubusercontent.com/state-spaces/mamba/main/"
               "mamba_ssm/ops/selective_scan_interface.py")
        s = urllib.request.urlopen(url, timeout=60).read().decode("utf-8")
        fn = next(n for n in _ast.parse(s).body
                  if isinstance(n, _ast.FunctionDef) and n.name == "selective_scan_ref")
        from einops import rearrange, repeat
        ns = {"torch": torch, "F": F, "rearrange": rearrange, "repeat": repeat}
        exec(_ast.get_source_segment(s, fn), ns)
        return ns["selective_scan_ref"], "resmi kaynaktan ayiklandi"
    except Exception as e:
        print(f"[BILGI] referans temin edilemedi: {type(e).__name__}: {e}")
        return None, None

def _operatorler(blk, x):
    """Taramaya kadar olan her sey (SelectiveSSMBlock.forward ile AYNI)."""
    B, L, _ = x.shape
    x_in, z = blk.in_proj(x).chunk(2, dim=-1)
    x_conv = F.silu(blk.conv1d(x_in.transpose(1, 2))[:, :, :L].transpose(1, 2))
    delta, Bm, Cm = torch.split(blk.x_proj(x_conv),
                                [blk.d_inner, blk.d_state, blk.d_state], dim=-1)
    delta = F.softplus(blk.dt_proj(delta))
    A = -torch.exp(blk.A_log)
    return x_conv, z, torch.exp(delta.unsqueeze(-1) * A), \
           delta.unsqueeze(-1) * Bm.unsqueeze(2), Bm, Cm, delta, A

def _tarama_dongu(blk, u, dA, dB, Cm):
    B, L, _ = u.shape
    h = torch.zeros(B, blk.d_inner, blk.d_state, device=u.device, dtype=u.dtype)
    ys = []
    for t in range(L):
        h = dA[:, t] * h + dB[:, t] * u[:, t].unsqueeze(-1)
        ys.append(torch.einsum("bds,bs->bd", h, Cm[:, t]))
    return torch.stack(ys, dim=1)

def validate_s6_block(d_model=64, d_state=16, B=4, L=64):
    """Uc kontrolu calistirir ve sonuclari sozluk olarak dondurur."""
    out = {}
    torch.manual_seed(0)
    ref, kaynak = _referans_scan()

    # ---- CHECK 1 ----
    if ref is not None:
        blk = SelectiveSSMBlock(d_model, d_state).to(DEVICE).eval()
        x = torch.randn(B, L, d_model, device=DEVICE)
        with torch.no_grad():
            u, z, dA, dB, Bm, Cm, delta, A = _operatorler(blk, x)
            bizim = _tarama_dongu(blk, u, dA, dB, Cm)
            r = ref(u.transpose(1, 2).contiguous(), delta.transpose(1, 2).contiguous(),
                    A, Bm.transpose(1, 2).contiguous(), Cm.transpose(1, 2).contiguous(),
                    D=None, z=None, delta_bias=None, delta_softplus=False,
                    return_last_state=False).transpose(1, 2)
            out["check1"] = float((bizim - r).abs().max().item())
        print(f"CHECK 1  max|bizim - selective_scan_ref| = {out['check1']:.3e}  "
              f"[{kaynak}]  (fp32 esik 1e-4)")
    else:
        out["check1"] = None
        print("CHECK 1  ATLANDI (referans temin edilemedi)")

    # ---- CHECK 2: kapali form, O(L^2), durum TASIMADAN ----
    blk64 = SelectiveSSMBlock(d_model, d_state).to(DEVICE).double().eval()
    x64 = torch.randn(B, L, d_model, device=DEVICE, dtype=torch.float64)
    with torch.no_grad():
        u, z, dA, dB, Bm, Cm, delta, A = _operatorler(blk64, x64)
        dongu = _tarama_dongu(blk64, u, dA, dB, Cm)
        src = dB * u.unsqueeze(-1)
        ys = []
        for t in range(L):
            acc = torch.zeros_like(src[:, 0])
            for s_ in range(t + 1):
                w = src[:, s_]
                if s_ < t:
                    w = w * dA[:, s_ + 1:t + 1].prod(dim=1)
                acc = acc + w
            ys.append(torch.einsum("bds,bs->bd", acc, Cm[:, t]))
        out["check2"] = float((dongu - torch.stack(ys, dim=1)).abs().max().item())
    print(f"CHECK 2  max|dongu - kapali form| = {out['check2']:.3e}  (fp64 esik 1e-8)")

    # ---- CHECK 3 ----
    kucuk = SelectiveSSMBlock(8, 4).to(DEVICE).double()
    xg = torch.randn(2, 6, 8, device=DEVICE, dtype=torch.float64, requires_grad=True)
    try:
        out["check3"] = bool(torch.autograd.gradcheck(
            lambda t: kucuk(t).sum(), (xg,), eps=1e-6, atol=1e-5, rtol=1e-3))
    except Exception as e:
        out["check3"] = False
        print(f"CHECK 3  HATA: {type(e).__name__}: {e}")
    print(f"CHECK 3  gradcheck (fp64, tam blok): {'GECTI' if out['check3'] else 'KALDI'}")
    return out

S6_VALIDATION = validate_s6_block()
with open(f"{PATHS['logs']}/s6_validation_{RUN_ID}.json", "w") as f:
    json.dump(S6_VALIDATION, f, indent=2)
print(f"\n[KAYDEDILDI] {PATHS['logs']}/s6_validation_{RUN_ID}.json")


## 8. Random Forest Referans Modeli (klasik ML karşılaştırması)

In [ ]:
# ============================================================
# HÜCRE 8 — RANDOM FOREST BASLANGIC MODELI (ortak arayuzle sarmalanmis)
# ============================================================
from sklearn.ensemble import RandomForestClassifier
import numpy as np
import torch

class RandomForestNIDS:
    """Torch modelleriyle AYNI .fit()/.predict_proba() arayuzunu sunan,
    sklearn tabanli klasik ML referans modeli (deep-learning-disi karsilastirma)."""
    def __init__(self, n_estimators: int = 200, seed: int = 42):
        self.model = RandomForestClassifier(
            n_estimators=n_estimators, max_depth=None, n_jobs=-1,
            random_state=seed, class_weight="balanced",
        )

    @staticmethod
    def _flatten(X: torch.Tensor) -> np.ndarray:
        return X.reshape(X.shape[0], -1).cpu().numpy()

    def fit(self, X_train: torch.Tensor, y_train: torch.Tensor):
        self.model.fit(self._flatten(X_train), y_train.cpu().numpy())
        return self

    def predict_proba(self, X: torch.Tensor) -> np.ndarray:
        return self.model.predict_proba(self._flatten(X))

    def predict(self, X: torch.Tensor) -> np.ndarray:
        return self.model.predict(self._flatten(X))

print("RandomForestNIDS tanimlandi (ortak arayuz: fit / predict / predict_proba).")


## 9. Eğitim Motoru

AMP (Otomatik Karışık Hassasiyet, A100 için bf16), checkpoint kaydet/devam-et,
erken durdurma ve Drive'a epoch-bazlı loglama içerir. Ayrıca isteğe bağlı
**FGSM-karışık adversarial training** modu içerir (`adv_training=True`) —
Hücre 14b'de savunma değerlendirmesi için kullanılır.

**Bilimsel doğruluk notları:** (1) Gerçek NIDS veri setleri (özellikle
NF-UNSW-NB15, ~%96 benign) ciddi sınıf dengesizliği içerdiğinden, kayıp
fonksiyonu ters-frekans **sınıf ağırlıklandırması** kullanır (RandomForest'taki
`class_weight='balanced'` ile tutarlı). (2) Model seçimi/erken durdurma, ham
doğruluk yerine **dengelenmiş doğruluk** (her sınıfın recall ortalaması)
kullanır — aksi halde dengesiz veride "her zaman çoğunluk sınıfını tahmin et"
gibi anlamsız bir model yanlışlıkla "en iyi" seçilebilirdi.


In [ ]:
# ============================================================
# HÜCRE 9 — EGITIM MOTORU (torch modelleri icin ortak)
# Ozellikler: AMP (bf16/A100), checkpoint kaydet/devam-et (Colab kopmalarina
# karsi dayaniklilik), erken durdurma, epoch-bazli Drive log dosyasi.
# ============================================================
import time, csv
import numpy as np
import torch
from torch.utils.data import TensorDataset, DataLoader

def make_loaders(X_train, y_train, X_test, y_test, batch_size: int, val_ratio: float = 0.1):
    n_val = int(len(X_train) * val_ratio)
    perm = torch.randperm(len(X_train))
    val_idx, tr_idx = perm[:n_val], perm[n_val:]
    train_ds = TensorDataset(X_train[tr_idx], y_train[tr_idx])
    val_ds = TensorDataset(X_train[val_idx], y_train[val_idx])
    test_ds = TensorDataset(X_test, y_test)
    g = torch.Generator().manual_seed(0)
    return (
        DataLoader(train_ds, batch_size=batch_size, shuffle=True, generator=g, drop_last=True),
        DataLoader(val_ds, batch_size=batch_size, shuffle=False),
        DataLoader(test_ds, batch_size=batch_size, shuffle=False),
    )

def checkpoint_path(dataset: str, model_name: str, seed: int, tag: str = "") -> str:
    # ONEMLI (v3 revizyonu): checkpoint dosya adina CONFIG['data_version'] eklenir. Boylece
    # bu REVIZYONDAN (grup-farkindali split) ONCEKI checkpoint'ler YANLISLIKLA 'tamamlanmis'
    # sayilip atlanmaz -- data_version degisince otomatik olarak YENIDEN egitim tetiklenir
    # (eski split ile egitilmis bir modelin yeni test setiyle sessizce degerlendirilmesini onler).
    #
    # HATA DUZELTMESI (v3.2): SMOKE_TEST durumu da etikete eklenir. Bunsuz, SMOKE_TEST=True
    # ile uretilen (3000 satir/3 epoch ile egitilmis, pratik olarak rastgele) bir checkpoint,
    # SMOKE_TEST=False'a gecildiginde AYNI dosya adiyla 'egitim tamamlanmis' sayilip
    # SESSIZCE yeniden kullanilir -- yani tam olcekli calisma, farkinda olmadan smoke-test
    # modelini sonuc olarak raporlar. Bu satir bunu ONLER: smoke ve tam-olcekli checkpoint'ler
    # HER ZAMAN farkli dosya adlarina sahip olur.
    tag_parts = [t for t in [tag, ("smoke" if CONFIG.get("smoke_test") else "")] if t]
    suffix = f"_{'_'.join(tag_parts)}" if tag_parts else ""
    return f"{PATHS['checkpoints']}/{dataset}_{model_name}_seed{seed}_{CONFIG['data_version']}{suffix}.pt"

def train_model(model_name: str, dataset: str, seed: int, X_train, y_train, X_test, y_test,
                 n_classes: int = 2, verbose: bool = True,
                 adv_training: bool = False, adv_eps: float = 0.05, adv_mix_ratio: float = 0.5):
    """Tek bir (veri seti, model, tohum) uclusu icin egitim. Checkpoint varsa ve
    egitim tamamlanmissa DOGRUDAN yuklenir (Colab oturum kopmasi sonrasi 'Run All'
    calistirildiginda tekrar egitim yapilmaz -> zaman/enerji tasarrufu ve kesintisizlik).

    adv_training=True ise, her egitim batch'inin (adv_mix_ratio) kadari, MEVCUT
    model agirliklariyla ANLIK olarak uretilen FGSM cekismeli ornekleriyle
    DEGISTIRILIR (Goodfellow ve ark. 2015 tarzi 'karisik adversarial training').
    Bu, Hucre 10'daki fgsm_attack() fonksiyonuna bagimlidir (calisma zamaninda
    cozulur; notebook'ta Hucre 10, Hucre 14/17'den ONCE calistirilmis olmalidir)."""
    seed_everything(seed)
    tag = "advtrain" if adv_training else ""
    ckpt_path = checkpoint_path(dataset, model_name, seed, tag=tag)

    if model_name == "RandomForest":
        if adv_training:
            print(f"[BILGI] RandomForest gradyan-tabanli degildir, adversarial training UYGULANAMAZ; "
                  f"standart egitimle devam ediliyor.")
        model = RandomForestNIDS(seed=seed)
        model.fit(X_train, y_train)
        return model, {"final_val_acc": None, "epochs_run": 0, "resumed": False}

    model = build_model(model_name, CONFIG["seq_len"], CONFIG["d_model"], CONFIG["n_layers"], n_classes).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG["learning_rate"], weight_decay=CONFIG["weight_decay"])
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CONFIG["epochs"])
    # ONEMLI (bilimsel dogruluk): Gercek NIDS veri setleri (ozellikle NF-UNSW-NB15,
    # ~%96 benign) CIDDI sinif dengesizligi icerir. Agirliksiz CrossEntropyLoss ile
    # az sayida epoch/hafif model, "her zaman benign de" seklinde ANLAMSIZ bir
    # sınıflandırıcıya yakinsayabilir (yuksek dogruluk, sifir gercek saldiri-tespit
    # yetenegi). Bu nedenle sinif agirliklari egitim setindeki GERCEK sinif
    # frekanslarindan hesaplanir (ters-frekans agirliklandirma, RandomForest'taki
    # class_weight='balanced' ile TUTARLI).
    class_counts = torch.bincount(y_train, minlength=n_classes).float()
    class_counts = torch.clamp(class_counts, min=1.0)  # sifir-bolme koruması
    class_weights = (class_counts.sum() / (n_classes * class_counts)).to(DEVICE)
    criterion = torch.nn.CrossEntropyLoss(weight=class_weights)
    if verbose:
        print(f"  [{dataset}/{model_name}/seed{seed}] sinif dagilimi (egitim)="
              f"{class_counts.cpu().tolist()}, sinif agirliklari={class_weights.cpu().tolist()}")

    if adv_training:
        adv_clip_min, adv_clip_max = get_feature_clip_bounds(X_train)

    start_epoch = 0
    best_val_acc = -1.0
    patience_counter = 0
    resumed = False

    if os.path.exists(ckpt_path):
        ckpt = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)
        model.load_state_dict(ckpt["model_state"])
        if ckpt.get("training_complete", False):
            print(f"[CHECKPOINT] {dataset}/{model_name}/seed{seed}{'/adv' if adv_training else ''} "
                  f"zaten TAMAMLANMIS, yeniden egitilmiyor.")
            return model, ckpt.get("meta", {"resumed": True})
        optimizer.load_state_dict(ckpt["optimizer_state"])
        start_epoch = ckpt["epoch"] + 1
        best_val_acc = ckpt.get("best_val_acc", -1.0)
        patience_counter = ckpt.get("patience_counter", 0)
        resumed = True
        print(f"[CHECKPOINT] {dataset}/{model_name}/seed{seed} epoch {start_epoch}'den DEVAM ediliyor.")

    train_loader, val_loader, _ = make_loaders(X_train, y_train, X_test, y_test, CONFIG["batch_size"])
    log_path = f"{PATHS['logs']}/train_{dataset}_{model_name}_seed{seed}{'_' + tag if tag else ''}.csv"
    log_mode = "a" if resumed and os.path.exists(log_path) else "w"
    log_file = open(log_path, log_mode, newline="")
    log_writer = csv.writer(log_file)
    if log_mode == "w":
        log_writer.writerow(["epoch", "train_loss", "val_loss", "val_balanced_acc", "seconds"])

    for epoch in range(start_epoch, CONFIG["epochs"]):
        t0 = time.time()
        model.train()
        total_loss = 0.0
        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE, non_blocking=True), yb.to(DEVICE, non_blocking=True)

            if adv_training:
                n_adv = int(xb.size(0) * adv_mix_ratio)
                if n_adv > 0:
                    # FGSM ornekleri, BatchNorm gibi katmanlarin egitim-modu
                    # istatistiklerini BOZMAMASI icin GECICI olarak eval moduna
                    # alinarak uretilir; ardindan asil egitim adimi icin train
                    # moduna geri donulur.
                    model.eval()
                    x_adv_part = fgsm_attack(
                        model, xb[:n_adv], yb[:n_adv], adv_eps, adv_clip_min, adv_clip_max
                    )
                    model.train()
                    xb = torch.cat([x_adv_part, xb[n_adv:]], dim=0)

            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type="cuda", dtype=AMP_DTYPE):
                logits = model(xb)
                loss = criterion(logits, yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            total_loss += loss.item() * xb.size(0)
        scheduler.step()
        train_loss = total_loss / len(train_loader.dataset)

        model.eval()
        val_loss, n = 0.0, 0
        val_preds, val_targets = [], []
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                with torch.autocast(device_type="cuda", dtype=AMP_DTYPE):
                    logits = model(xb)
                    loss = criterion(logits, yb)
                val_loss += loss.item() * xb.size(0)
                val_preds.append(logits.argmax(-1).cpu())
                val_targets.append(yb.cpu())
                n += xb.size(0)
        val_loss /= n
        val_preds = torch.cat(val_preds)
        val_targets = torch.cat(val_targets)
        # DENGELENMIS dogruluk (her sinifin recall'inin ortalamasi) kullanilir:
        # ham dogruluk, ciddi sinif dengesizliginde (orn. NIDS'te %95+ benign)
        # 'her zaman cogunluk sinifini tahmin et' gibi ANLAMSIZ bir modele
        # yuksek puan verebilir. Bu, hem erken durdurma hem de checkpoint secimi
        # icin RandomForest'taki class_weight='balanced' ile TUTARLI bir kriterdir.
        per_class_recalls = []
        for cls in range(n_classes):
            cls_mask = val_targets == cls
            if cls_mask.sum() > 0:
                per_class_recalls.append((val_preds[cls_mask] == cls).float().mean().item())
        val_acc = float(np.mean(per_class_recalls)) if per_class_recalls else 0.0
        dt = time.time() - t0
        log_writer.writerow([epoch, train_loss, val_loss, val_acc, dt])
        log_file.flush()

        improved = val_acc > best_val_acc
        if improved:
            best_val_acc = val_acc
            patience_counter = 0
        else:
            patience_counter += 1

        torch.save({
            "model_state": model.state_dict(),
            "optimizer_state": optimizer.state_dict(),
            "epoch": epoch,
            "best_val_acc": best_val_acc,
            "patience_counter": patience_counter,
            "training_complete": False,
            "meta": {"final_val_acc": best_val_acc, "epochs_run": epoch + 1, "resumed": resumed},
        }, ckpt_path)

        if verbose and (epoch % 5 == 0 or epoch == CONFIG["epochs"] - 1):
            print(f"  [{dataset}/{model_name}/seed{seed}] epoch {epoch+1}/{CONFIG['epochs']} "
                  f"train_loss={train_loss:.4f} val_loss={val_loss:.4f} val_balanced_acc={val_acc:.4f} ({dt:.1f}s)")

        if patience_counter >= CONFIG["patience"]:
            print(f"  [{dataset}/{model_name}/seed{seed}] erken durdurma (epoch {epoch+1}).")
            break

    log_file.close()
    # En iyi durumu tamamlanmis olarak isaretle
    ckpt = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)
    ckpt["training_complete"] = True
    torch.save(ckpt, ckpt_path)
    model.load_state_dict(ckpt["model_state"])
    return model, ckpt["meta"]

print("Egitim motoru hazir: train_model(model_name, dataset, seed, ...)")


## 9b. Çıkarım Hızı (Latency/Throughput) Ölçüm Modülü

Mamba/SSM mimarilerinin literatürdeki temel iddiası "doğrusal karmaşıklık
sayesinde daha hızlı çıkarım"dır. Bu iddia burada **gerçek ölçülmüş** gecikme
(ms/örnek) ve verim (örnek/saniye) rakamlarıyla sınanır — sadece parametre
sayısıyla değil.

> **Şeffaflık notu:** Bizim S6 yeniden-uygulamamız CUDA derlemesi gerektirmemek
> için dizi boyutunda bir Python for-döngüsü kullanır; bu nedenle burada ölçülen
> rakamlar resmi `mamba-ssm` paketinin paralel-tarama CUDA çekirdeklerinden
> farklı olabilir (bkz. hücre içi ayrıntılı not).


In [ ]:
# ============================================================
# HÜCRE 9b — CIKARIM HIZI (LATENCY / THROUGHPUT) OLCUM MODULU
#
# GEREKCE: Mamba/SSM mimarilerinin literaturdeki temel iddiasi "dogrusal
# karmasiklik sayesinde Transformer'a gore daha hizli cikarim" seklindedir
# (bkz. arastirma gap analizi). Bu iddia, SADECE parametre sayisiyla degil,
# GERCEK OLCULMUS gecikme (latency) ve verim (throughput) rakamlariyla
# kanitlanmalidir -- aksi halde makale bu iddiayi one surup DOGRULAMAMIS olur.
# ONEMLI SEFFAFLIK NOTU: Hucre 7'deki Mamba (S6) yeniden uygulamasi, CUDA
# derlemesi GEREKTIRMEMEK icin dizi (sequence) boyutu uzerinde bir PYTHON
# FOR-DONGUSU kullanir (bkz. Hucre 7 yorumu). Bu, TEKRARLANABILIRLIK ve
# "kesintisiz calisma" hedefleri icin BILINCLI bir mimari tercihtir; ANCAK
# resmi 'mamba-ssm' paketinin ozel paralel-tarama CUDA cekirdekleriyle elde
# ettigi dogrusal-zaman verimliligini BIREBIR YANSITMAYABILIR. Bu hucrede
# olculen gecikme/verim rakamlari, SADECE bu calismadaki saf-PyTorch yeniden
# uygulamaya aittir ve makalede bu sekilde ACIKCA belirtilmelidir (bkz. Hucre 16
# Final Rapor, "Onemli Metodolojik Notlar" bolumu).
# ============================================================
import time
import numpy as np
import pandas as pd
import torch

@torch.no_grad()
def measure_inference_latency(model, seq_len: int, device, model_name: str,
                                batch_sizes=(1, 512), n_warmup: int = 20, n_repeats: int = 100) -> dict:
    """Verilen model icin, farkli batch buyukluklerinde gecikme (ms/ornek) ve
    verim (ornek/saniye) olcer. GPU icin dogru zamanlama saglamak amaciyla
    torch.cuda.synchronize() KULLANILIR (senkronize edilmeden yapilan olcumler
    GPU'nun asenkron yurutmesi nedeniyle YANLIS/COK-IYIMSER sonuc verir)."""
    results = {}
    is_cuda = device.type == "cuda"

    if model_name == "RandomForest":
        # RF, sklearn ile CPU uzerinde calisir; GPU modelleriyle DOGRUDAN
        # karsilastirilamaz ama referans olarak raporlanir (bkz. makale notu).
        for bs in batch_sizes:
            X_dummy = torch.randn(bs, seq_len, 1)
            X_flat = model._flatten(X_dummy)
            for _ in range(min(5, n_warmup)):
                model.model.predict(X_flat)
            t0 = time.perf_counter()
            for _ in range(min(20, n_repeats)):
                model.model.predict(X_flat)
            elapsed = time.perf_counter() - t0
            n_runs = min(20, n_repeats)
            ms_per_batch = (elapsed / n_runs) * 1000
            results[bs] = {
                "latency_ms_per_batch": ms_per_batch,
                "latency_ms_per_sample": ms_per_batch / bs,
                "throughput_samples_per_sec": bs * n_runs / elapsed,
                "device": "cpu (sklearn, agac-tabanli)",
            }
        return results

    model.eval()
    model.to(device)
    for bs in batch_sizes:
        X_dummy = torch.randn(bs, seq_len, 1, device=device)

        # Isinma (warmup) - CUDA kernel derlemesi/cache etkisini olcumden CIKARIR
        for _ in range(n_warmup):
            _ = model(X_dummy)
        if is_cuda:
            torch.cuda.synchronize()

        if is_cuda:
            timings = []
            for _ in range(n_repeats):
                start_evt = torch.cuda.Event(enable_timing=True)
                end_evt = torch.cuda.Event(enable_timing=True)
                start_evt.record()
                _ = model(X_dummy)
                end_evt.record()
                torch.cuda.synchronize()
                timings.append(start_evt.elapsed_time(end_evt))  # ms
            ms_per_batch = float(np.mean(timings))
            ms_std = float(np.std(timings))
        else:
            t0 = time.perf_counter()
            for _ in range(n_repeats):
                _ = model(X_dummy)
            ms_per_batch = (time.perf_counter() - t0) / n_repeats * 1000
            ms_std = float("nan")

        results[bs] = {
            "latency_ms_per_batch": ms_per_batch,
            "latency_ms_per_batch_std": ms_std,
            "latency_ms_per_sample": ms_per_batch / bs,
            "throughput_samples_per_sec": bs / (ms_per_batch / 1000),
            "device": str(device),
        }
    return results


def build_efficiency_table(param_counts: dict, latency_results: dict, batch_sizes=(1, 512)) -> pd.DataFrame:
    """Tum modeller icin parametre sayisi + gecikme/verim rakamlarini TEK bir
    'verimlilik' tablosunda birlestirir (makalenin Tartisma bolumu icin)."""
    rows = []
    for model_name, res in latency_results.items():
        row = {"model": model_name, "n_parameters": param_counts.get(model_name, np.nan)}
        for bs in batch_sizes:
            if bs in res:
                row[f"latency_ms_bs{bs}"] = res[bs]["latency_ms_per_sample"]
                row[f"throughput_sps_bs{bs}"] = res[bs]["throughput_samples_per_sec"]
        rows.append(row)
    df = pd.DataFrame(rows)
    if f"latency_ms_bs{batch_sizes[0]}" in df.columns:
        df = df.sort_values(f"latency_ms_bs{batch_sizes[0]}")
    return df

print("Cikarim hizi olcum modulu hazir: measure_inference_latency(), build_efficiency_table().")


## 10. Çekişmeli (Adversarial) Saldırı Modülü

**Önemli:** FGSM ve PGD, popüler `torchattacks` kütüphanesinin görüntü-verisi
varsayımıyla geldiği `[0,1]` clamp sınırlamasından kaçınmak için **özel olarak
sıfırdan** (Goodfellow ve ark. 2015; Madry ve ark. 2018 formülasyonlarına sadık
kalınarak) uygulanmıştır. Bu, standartlaştırılmış (negatif değerler içeren) tablo
verisi üzerinde **doğru ve tekrarlanabilir** sonuçlar sağlar.


In [ ]:
# ============================================================
# HÜCRE 10 — CEKISMELI (ADVERSARIAL) SALDIRI MODULU
#
# ONEMLI METODOLOJIK NOT (hakem degerlendirmesi icin):
# Yaygin kullanilan 'torchattacks' kutuphanesinin FGSM/PGD uygulamalari,
# goruntu verisi varsayimiyla cikti pertürbasyonunu SABIT [0,1] araligina
# clamp'lemektedir (bkz. kutuphane kaynak kodu). Bizim verimiz StandardScaler
# ile olceklendirilmis (negatif degerler icerebilen, [0,1] disinda) tablo/
# NetFlow verisi oldugundan, bu sabit varsayim SESSIZCE HATALI sonuclara yol
# acar. Bu nedenle FGSM ve PGD, asagida veri-dagilimina-duyarli (clip_min,
# clip_max parametreli) ve Linf-norm formulasyonuna SADIK kalacak sekilde
# SIFIRDAN, saf PyTorch ile yeniden uygulanmistir (Goodfellow et al., 2015;
# Madry et al., 2018). Bu, hem seffafligi hem de tekrarlanabilirligi saglar.
# ============================================================
import torch
import torch.nn as nn
import torch.nn.functional as F

class ModelWrapper(nn.Module):
    """Saldiri sirasinda modelin GERCEKCI ozellik araligini (egitim
    dagiliminin gozlenen min/maks degerleri) asmamasini saglar."""
    def __init__(self, model: nn.Module, clip_min: float, clip_max: float):
        super().__init__()
        self.model = model
        self.clip_min = clip_min
        self.clip_max = clip_max

    def forward(self, x):
        return self.model(torch.clamp(x, self.clip_min, self.clip_max))


def get_feature_clip_bounds(X_train: torch.Tensor):
    """Egitim dagiliminin gozlenen min/maks sinirlarini dondurur; boylece
    uretilen cekismeli ornekler ISTATISTIKSEL OLARAK GECERLI (gercekci)
    ozellik degerleri araliginda kalir."""
    return float(X_train.min().item()), float(X_train.max().item())


def fgsm_attack(model: nn.Module, x: torch.Tensor, y: torch.Tensor, eps: float,
                 clip_min: float, clip_max: float) -> torch.Tensor:
    """Fast Gradient Sign Method (Goodfellow et al., 2015). Linf-norm, tek adim.

    HATA DUZELTMESI (v3.1): LSTM vekil/hedef modeller icin, model eval() modundeyken
    cuDNN'in RNN implementasyonu backward-pass icin gereken ara durumu SAKLAMAZ ve
    'cudnn RNN backward can only be called in training mode' hatasi firlatir. Bu, tum
    mimarilerde ORTAK kullanilan fonksiyonda, gradyan hesaplanan forward/backward
    blogu cuDNN devre disi birakilarak (PyTorch'un native/genel RNN implementasyonuna
    dusurulerek) calistirilir -- SADECE bu blok icin; sayisal sonucu ETKILEMEZ, sadece
    hangi kernel'in kullanildigini degistirir."""
    x = x.clone().detach().requires_grad_(True)
    with torch.backends.cudnn.flags(enabled=False):
        logits = model(x)
        loss = F.cross_entropy(logits, y)
        grad = torch.autograd.grad(loss, x)[0]
    x_adv = x + eps * grad.sign()
    return torch.clamp(x_adv, clip_min, clip_max).detach()


def pgd_attack(model: nn.Module, x: torch.Tensor, y: torch.Tensor, eps: float,
                alpha: float, steps: int, clip_min: float, clip_max: float,
                random_start: bool = True) -> torch.Tensor:
    """Projected Gradient Descent (Madry et al., 2018). Linf-norm, coklu adim,
    her adimda [x0-eps, x0+eps] kupune PROJEKSIYON uygulanir."""
    x0 = x.clone().detach()
    if random_start:
        x_adv = x0 + torch.empty_like(x0).uniform_(-eps, eps)
        x_adv = torch.clamp(x_adv, clip_min, clip_max).detach()
    else:
        x_adv = x0.clone().detach()

    for _ in range(steps):
        x_adv.requires_grad_(True)
        # HATA DUZELTMESI (v3.1): bkz. fgsm_attack() ustundeki not -- LSTM icin cuDNN
        # RNN backward, eval() modunda calismaz; bu blok icin cuDNN devre disi birakilir.
        with torch.backends.cudnn.flags(enabled=False):
            logits = model(x_adv)
            loss = F.cross_entropy(logits, y)
            grad = torch.autograd.grad(loss, x_adv)[0]
        with torch.no_grad():
            x_adv = x_adv + alpha * grad.sign()
            perturbation = torch.clamp(x_adv - x0, -eps, eps)   # Linf projeksiyonu
            x_adv = torch.clamp(x0 + perturbation, clip_min, clip_max)
    return x_adv.detach()


def generate_adversarial_examples(model: nn.Module, X: torch.Tensor, y: torch.Tensor,
                                   attack_type: str, eps: float, clip_min: float, clip_max: float,
                                   device, batch_size: int = 256, pgd_alpha: float = None,
                                   pgd_steps: int = 10) -> torch.Tensor:
    """Bellek-dostu, batch'ler halinde cekismeli ornek uretimi (A100 icin guvenli,
    tum veri seti tek seferde GPU belegine yuklenmez)."""
    model.eval()
    wrapped = ModelWrapper(model, clip_min, clip_max).to(device)
    adv_batches = []
    for i in range(0, len(X), batch_size):
        xb = X[i:i + batch_size].to(device)
        yb = y[i:i + batch_size].to(device)
        if attack_type == "FGSM":
            adv = fgsm_attack(wrapped, xb, yb, eps, clip_min, clip_max)
        elif attack_type == "PGD":
            alpha = pgd_alpha if pgd_alpha is not None else eps / 4
            adv = pgd_attack(wrapped, xb, yb, eps, alpha, pgd_steps, clip_min, clip_max)
        else:
            raise ValueError(f"Bilinmeyen saldiri tipi: {attack_type}")
        adv_batches.append(adv.detach().cpu())
    return torch.cat(adv_batches)


@torch.no_grad()
def _predict(model: nn.Module, X: torch.Tensor, device, batch_size: int = 512) -> torch.Tensor:
    model.eval()
    preds = []
    for i in range(0, len(X), batch_size):
        xb = X[i:i + batch_size].to(device)
        preds.append(model(xb).argmax(-1).cpu())
    return torch.cat(preds)


def blackbox_transfer_attack(surrogate_model: nn.Module, target_model: nn.Module,
                              X: torch.Tensor, y: torch.Tensor, eps: float,
                              clip_min: float, clip_max: float, device,
                              pgd_steps: int = 10) -> dict:
    """GERCEK kara-kutu (black-box) senaryosu: saldirgan SADECE bagimsiz bir
    vekil (surrogate) modele erisir; hedef modelin agirliklarini/mimarisini
    GORMEZ. Vekil uzerinde PGD ile uretilen ornekler hedef modele 'aktarilir'
    (transferability, Papernot et al., 2017) ve hedefin dogrulugundaki dusus
    olculur -> gercek dunyadaki en gerceci tehdit modeli."""
    X_adv = generate_adversarial_examples(
        surrogate_model, X, y, "PGD", eps, clip_min, clip_max, device,
        pgd_alpha=eps / 4, pgd_steps=pgd_steps,
    )
    preds_clean = _predict(target_model, X, device)
    preds_adv = _predict(target_model, X_adv, device)
    clean_acc = (preds_clean == y).float().mean().item()
    adv_acc = (preds_adv == y).float().mean().item()
    attack_success = 1.0 - (adv_acc / clean_acc) if clean_acc > 1e-8 else 0.0
    return {"clean_acc": clean_acc, "adv_acc": adv_acc, "attack_success_rate": attack_success}

print("Cekismeli saldiri modulu hazir: ozel FGSM, PGD ve Black-Box Transfer (Linf-norm, veri-dagilimina-duyarli clip).")


## 11. Değerlendirme Modülü

Temiz + sağlam metrikler, istatistiksel anlamlılık testleri (eşli t-testi +
Wilcoxon), ikili sınıf-bazlı kırılganlık VE **saldırı alt-türü bazlı**
kırılganlık analizi (`compute_attack_category_vulnerability`) — "hangi saldırı
türü bu mimariyi en çok kandırıyor?" sorusunun doğrudan yanıtı.


In [ ]:
# ============================================================
# HÜCRE 11 — DEGERLENDIRME MODULU
# Temiz performans + saldiri-altinda (robust) performans + sinif-bazli
# kirilganlik + coklu-tohum istatistiksel anlamlilik testleri.
# ============================================================
import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support, roc_auc_score, confusion_matrix,
)
from scipy import stats

@torch.no_grad()
def _get_probs_torch(model, X: torch.Tensor, device, batch_size=512) -> np.ndarray:
    model.eval()
    probs = []
    for i in range(0, len(X), batch_size):
        xb = X[i:i + batch_size].to(device)
        p = torch.softmax(model(xb), dim=-1)
        probs.append(p.cpu().numpy())
    return np.concatenate(probs)

def get_probs(model, X, device, model_name: str) -> np.ndarray:
    if model_name == "RandomForest":
        return model.predict_proba(X)
    return _get_probs_torch(model, X, device)

def compute_metrics(y_true: np.ndarray, probs: np.ndarray) -> dict:
    y_pred = probs.argmax(axis=-1)
    acc = accuracy_score(y_true, y_pred)
    prec, rec, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro", zero_division=0
    )
    try:
        auc = roc_auc_score(y_true, probs[:, 1])
    except Exception:
        auc = float("nan")
    return {"accuracy": acc, "precision_macro": prec, "recall_macro": rec,
            "f1_macro": f1, "auc": auc}

def compute_per_class_vulnerability(y_true: np.ndarray, probs_clean: np.ndarray,
                                     probs_adv: np.ndarray) -> pd.DataFrame:
    pred_clean = probs_clean.argmax(axis=-1)
    pred_adv = probs_adv.argmax(axis=-1)
    rows = []
    for cls in np.unique(y_true):
        mask = y_true == cls
        if mask.sum() == 0:
            continue
        clean_acc = (pred_clean[mask] == cls).mean()
        adv_acc = (pred_adv[mask] == cls).mean()
        rows.append({
            "class": int(cls), "n_samples": int(mask.sum()),
            "clean_acc": clean_acc, "adv_acc": adv_acc,
            "acc_drop": clean_acc - adv_acc,
        })
    return pd.DataFrame(rows)

def compute_attack_category_vulnerability(y_true: np.ndarray, attack_category: np.ndarray,
                                           probs_clean: np.ndarray, probs_adv: np.ndarray,
                                           min_samples: int = 5) -> pd.DataFrame:
    """SALDIRI ALT-TURU bazinda kirilganlik: 'Bu mimari hangi saldiri kategorisine
    (DoS, Probe, Exploits, Reconnaissance, vb.) karsi ozellikle savunmasiz?' sorusunu
    yanitlar. Sadece GERCEKTEN saldiri olan (binary_label==1) ornekler dahil edilir
    -- Benign sinifin 'kandirilmasi' zaten genel dogruluk metriklerinde yer alir.
    Bir kategorinin dogru tespit edilmesi = model o ornegi SALDIRI (1) olarak
    siniflandirir; 'atlatma basarisi' (evasion) = saldirinin BENIGN sanilmasidir."""
    pred_clean = probs_clean.argmax(axis=-1)
    pred_adv = probs_adv.argmax(axis=-1)
    rows = []
    attack_mask = y_true == 1
    for cat in sorted(set(attack_category[attack_mask].tolist())):
        if cat == "Benign":
            continue
        mask = attack_mask & (attack_category == cat)
        n = int(mask.sum())
        if n < min_samples:
            continue  # istatistiksel olarak anlamsiz derecede az ornek -> disla
        clean_detection = (pred_clean[mask] == 1).mean()   # saldiri olarak DOGRU tespit orani
        adv_detection = (pred_adv[mask] == 1).mean()
        rows.append({
            "attack_category": cat, "n_samples": n,
            "clean_detection_rate": clean_detection,
            "adv_detection_rate": adv_detection,
            "evasion_success_rate": clean_detection - adv_detection,  # ne kadar dustu
        })
    df = pd.DataFrame(rows)
    if not df.empty:
        df = df.sort_values("evasion_success_rate", ascending=False).reset_index(drop=True)
    return df

def paired_significance_test(scores_a: list, scores_b: list) -> dict:
    """Coklu-tohum sonuclari arasinda esli t-testi (Mamba vs. en iyi baseline).
    n < 30 oldugu icin ayrica Wilcoxon isaretli-sira testi de raporlanir."""
    if len(scores_a) < 2 or len(scores_b) < 2:
        return {"t_stat": None, "p_value": None, "wilcoxon_p": None, "note": "yetersiz tohum sayisi"}
    t_stat, p_val = stats.ttest_rel(scores_a, scores_b)
    try:
        w_stat, w_p = stats.wilcoxon(scores_a, scores_b)
    except Exception:
        w_p = None
    # REVIZYON: Cohen'in d_z ETKI BUYUKLUGU. p-degeri, etki buyuklugu ile
    # tekrar sayisini birbirine karistirir; d_z tohum sayisindan BAGIMSIZDIR
    # ve coklu-karsilastirma duzeltmesinden ETKILENMEZ.
    diffs = np.asarray(scores_a, dtype=float) - np.asarray(scores_b, dtype=float)
    d_z = float(diffs.mean() / diffs.std(ddof=1)) if diffs.std(ddof=1) > 0 else float("nan")
    return {"t_stat": float(t_stat), "p_value": float(p_val),
            "wilcoxon_p": w_p, "cohens_dz": d_z}

def holm_bonferroni_correction(p_values: list, alpha: float = 0.05) -> list:
    """Holm-Bonferroni ADIM-ASAGI (step-down) coklu-karsilastirma duzeltmesi.

    GEREKCE (v3 metodolojik revizyonu -- hakem geri bildirimi): Notebook, her (veri seti
    x saldiri) kombinasyonu icin AYRI bir esli t-testi/Wilcoxon calistirir; bu COK SAYIDA
    hipotez testi duzeltilmeden raporlanirsa aile-capinda Tip-I hata (yanlis-pozitif
    'anlamli fark') orani SISER. Holm-Bonferroni, duz Bonferroni'den DAHA GUCLU (daha az
    muhafazakar) ama yine de aile-capinda hata oranini (FWER) kontrol eden standart bir
    yontemdir. Girdi siraligina karsilik gelen DUZELTILMIS p-degerleri listesini dondurur
    (None/NaN girdiler oldugu gibi korunur)."""
    idx_valid = [i for i, p in enumerate(p_values) if p is not None and not (isinstance(p, float) and np.isnan(p))]
    if not idx_valid:
        return list(p_values)
    m = len(idx_valid)
    order = sorted(idx_valid, key=lambda i: p_values[i])
    corrected = [None] * len(p_values)
    running_max = 0.0
    for rank, i in enumerate(order):
        adj = (m - rank) * p_values[i]
        running_max = max(running_max, adj)
        corrected[i] = min(1.0, running_max)
    for i, p in enumerate(p_values):
        if i not in idx_valid:
            corrected[i] = p
    return corrected

print("Degerlendirme modulu hazir: compute_metrics, compute_per_class_vulnerability, "
      "paired_significance_test, holm_bonferroni_correction.")


## 12. Yayına-Hazır Grafik Üretimi (300dpi PNG + vektör PDF)

Temiz doğruluk karşılaştırması, sağlamlık eğrileri, ikili ve saldırı-alt-türü
bazlı kırılganlık grafikleri, ölçülen-gecikme vs. sağlamlık grafiği, savunma
etkinliği karşılaştırması ve hiperparametre ablasyon grafiği dahil.


In [ ]:
# ============================================================
# REVIZYON: makale Ingilizce oldugundan TUM grafik etiketleri Ingilizce'dir.
# Ayrica yinelemeli baseline BiLSTM olarak adlandirilir (cift yonlu LSTM).
DISPLAY_NAMES = {"LSTM": "BiLSTM", "RandomForest": "Random Forest"}
def disp(m):
    return DISPLAY_NAMES.get(m, m)

# ============================================================
# HÜCRE 12 — YAYINA-HAZIR ("publication-ready") GRAFIK URETIMI
# Tum grafikler 300dpi PNG (Word/PowerPoint icin) + vektor PDF (LaTeX icin)
# olarak Drive/figures altina kaydedilir. BTD ve genel dergi standartlarina
# uygun sade, yuksek-kontrastli, renk-korlugune duyarli bir palet kullanilir.
# ============================================================
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import pandas as pd
import numpy as np
from sklearn.metrics import confusion_matrix

PUB_PALETTE = ["#1b4965", "#c9184a", "#2a9d8f", "#e9c46a", "#6a4c93"]

def set_publication_style():
    plt.rcParams.update({
        "figure.dpi": 150,
        "savefig.dpi": 300,
        "font.family": "serif",
        "font.size": 11,
        "axes.titlesize": 13,
        "axes.titleweight": "bold",
        "axes.labelsize": 11,
        "legend.fontsize": 9.5,
        "xtick.labelsize": 9.5,
        "ytick.labelsize": 9.5,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.grid": True,
        "grid.alpha": 0.25,
        "grid.linestyle": "--",
        "figure.constrained_layout.use": True,
    })
    sns.set_palette(PUB_PALETTE)

def _save(fig, name: str):
    # HATA DUZELTMESI (v3.3): PNG kaydi HER ZAMAN denenir ve basarisiz olursa
    # istisna yukselir (bu, sonuclarin gorsellestirilmesi icin KRITIK). PDF
    # (vektor) kaydi ise bazi Colab ortamlarinda matplotlib'in PDF backend'i
    # (backend_pdf -> font_manager.FontPath) bozuk/versiyon-uyumsuz olabildigi
    # icin OPSIYONEL sayilir: basarisiz olursa sadece UYARI verilir, calisma
    # DEVAM eder. Boylece saatler surebilecek bir pipeline, sirf kozmetik bir
    # vektor-format dısa aktarma sorunu yuzunden BASTAN SONA cokmez.
    png_path = f"{PATHS['figures']}/{name}.png"
    pdf_path = f"{PATHS['figures']}/{name}.pdf"
    fig.savefig(png_path, dpi=300, bbox_inches="tight")
    try:
        fig.savefig(pdf_path, bbox_inches="tight")
        print(f"[GRAFIK KAYDEDILDI] {png_path}  |  {pdf_path}")
    except Exception as e:
        pdf_path = None
        print(f"[UYARI] {name}: PNG kaydedildi ama PDF (vektor) kaydi basarisiz oldu "
              f"(ortam kaynakli matplotlib PDF backend sorunu olabilir): {type(e).__name__}: {e}")
        print(f"[GRAFIK KAYDEDILDI] {png_path}  (PDF ATLANDI)")
    plt.close(fig)
    return png_path, pdf_path

def plot_clean_accuracy_bar(results_df: pd.DataFrame, dataset: str):
    """Veri seti icin, model basina ortalama temiz dogruluk (+/- std, coklu tohum)."""
    set_publication_style()
    sub = results_df[(results_df["dataset"] == dataset) & (results_df["attack"] == "clean")]
    agg = sub.groupby("model")["accuracy"].agg(["mean", "std"]).reset_index().sort_values("mean", ascending=False)
    fig, ax = plt.subplots(figsize=(6, 4))
    bars = ax.bar(agg["model"], agg["mean"], yerr=agg["std"].fillna(0), capsize=4,
                   color=PUB_PALETTE[:len(agg)], edgecolor="black", linewidth=0.6)
    for b, v in zip(bars, agg["mean"]):
        ax.text(b.get_x() + b.get_width() / 2, v + 0.01, f"{v:.3f}", ha="center", fontsize=9)
    ax.set_ylim(0, 1.08)
    ax.set_ylabel("Test Dogrulugu (Accuracy)")
    ax.set_title(f"{dataset}: Temiz Veride Model Karsilastirmasi (coklu tohum ortalamasi)")
    return _save(fig, f"clean_accuracy_{dataset}")

def plot_robustness_curve(results_df: pd.DataFrame, dataset: str, attack: str):
    """Epsilon'a (pertürbasyon buyuklugune) karsi dogruluk dususu egrisi - modeller karsilastirmali."""
    set_publication_style()
    sub = results_df[(results_df["dataset"] == dataset) & (results_df["attack"] == attack)]
    if sub.empty:
        return None
    fig, ax = plt.subplots(figsize=(6.5, 4.2))
    for i, model in enumerate(sorted(sub["model"].unique())):
        m_sub = sub[sub["model"] == model].groupby("eps")["accuracy"].agg(["mean", "std"]).reset_index()
        ax.errorbar(m_sub["eps"], m_sub["mean"], yerr=m_sub["std"].fillna(0),
                     marker="o", capsize=3, label=disp(model), color=PUB_PALETTE[i % len(PUB_PALETTE)], linewidth=1.8)
    ax.set_xlabel(r"Pertürbasyon Buyuklugu ($\epsilon$, $L_\infty$)")
    ax.set_ylabel("Robust accuracy")
    ax.set_title(f"{dataset} -- robustness under {attack}")
    ax.legend(frameon=False)
    ax.set_ylim(0, 1.05)
    return _save(fig, f"robustness_curve_{dataset}_{attack}")

def plot_per_class_heatmap(vuln_df: pd.DataFrame, dataset: str, model: str, attack: str):
    """Sinif-bazli dogruluk dususunu isi haritasi olarak gosterir."""
    set_publication_style()
    fig, ax = plt.subplots(figsize=(5.5, max(2.5, 0.4 * len(vuln_df))))
    data = vuln_df.set_index("class")[["clean_acc", "adv_acc"]]
    sns.heatmap(data.T, annot=True, fmt=".2f", cmap="RdYlGn", vmin=0, vmax=1,
                cbar_kws={"label": "Accuracy"}, linewidths=0.5, ax=ax)
    ax.set_title(f"{dataset} / {model}: {attack} Altinda Sinif-Bazli Kirilganlik")
    ax.set_xlabel("Sinif")
    return _save(fig, f"per_class_vuln_{dataset}_{model}_{attack}")

def plot_attack_category_vulnerability(cat_vuln_df: pd.DataFrame, dataset: str, model: str, attack: str):
    """SALDIRI ALT-TURU bazinda 'atlatma basari orani' (evasion success rate)
    yatay cubuk grafigi -- 'hangi saldiri turu bu mimariyi en cok kandiriyor?'
    sorusunun dogrudan gorsel yaniti. Kategoriler evasion oranina gore siralanir."""
    if cat_vuln_df is None or cat_vuln_df.empty:
        return None
    set_publication_style()
    df = cat_vuln_df.sort_values("evasion_success_rate", ascending=True)
    fig, ax = plt.subplots(figsize=(6.5, max(2.8, 0.45 * len(df))))
    colors = ["#c9184a" if v > 0.5 else ("#e9c46a" if v > 0.2 else "#2a9d8f")
              for v in df["evasion_success_rate"]]
    bars = ax.barh(df["attack_category"], df["evasion_success_rate"], color=colors,
                    edgecolor="black", linewidth=0.6)
    for b, v, n in zip(bars, df["evasion_success_rate"], df["n_samples"]):
        ax.text(v + 0.01, b.get_y() + b.get_height() / 2, f"{v:.2f} (n={n})",
                va="center", fontsize=8.5)
    ax.set_xlim(0, 1.15)
    ax.set_xlabel("Atlatma (Evasion) Basari Orani")
    ax.set_title(f"{dataset} / {model}: {attack} Altinda Saldiri-Turu Bazli Kirilganlik")
    return _save(fig, f"category_vuln_{dataset}_{model}_{attack}")

def plot_confusion_matrix(y_true, y_pred, dataset: str, model: str, tag: str):
    set_publication_style()
    cm = confusion_matrix(y_true, y_pred)
    fig, ax = plt.subplots(figsize=(4.2, 3.8))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False, ax=ax)
    ax.set_xlabel("Tahmin Edilen")
    ax.set_ylabel("Gercek")
    ax.set_title(f"{dataset} / {model} ({tag})")
    return _save(fig, f"confusion_{dataset}_{model}_{tag}")

def plot_params_vs_robust_accuracy(results_df: pd.DataFrame, param_counts: dict):
    """Verimlilik-sagitmlik odakli 'publication' grafik: parametre sayisi vs.
    ortalama sağlam dogruluk (Mamba'nin verimlilik iddiasini gorsellestirir)."""
    set_publication_style()
    adv = results_df[results_df["attack"] != "clean"]
    agg = adv.groupby("model")["accuracy"].mean().reset_index()
    agg["params"] = agg["model"].map(param_counts)
    agg = agg.dropna()
    fig, ax = plt.subplots(figsize=(6, 4.2))
    for i, row in agg.iterrows():
        ax.scatter(row["params"], row["accuracy"], s=140, color=PUB_PALETTE[i % len(PUB_PALETTE)],
                   edgecolor="black", zorder=3)
        ax.annotate(disp(row["model"]), (row["params"], row["accuracy"]),
                     textcoords="offset points", xytext=(8, 4), fontsize=10)
    ax.set_xscale("log")
    ax.set_xlabel("Model Parametre Sayisi (log olcek)")
    ax.set_ylabel("Mean robust accuracy (all attacks)")
    ax.set_title("Parameter count versus robustness")
    return _save(fig, "params_vs_robust_accuracy")

def plot_latency_vs_robust_accuracy(results_df: pd.DataFrame, efficiency_df: pd.DataFrame, batch_size: int = 1):
    """GERCEK OLCULMUS cikarim gecikmesi (ms/ornek) vs. ortalama sağlam dogruluk.
    Bu grafik, sadece parametre sayisina degil, FIILEN OLCULEN hiza dayandigi
    icin verimlilik iddialari acisindan parametre-sayisi grafiginden DAHA
    ikna edicidir (bkz. Hucre 9b metodolojik seffaflik notu)."""
    set_publication_style()
    adv = results_df[results_df["attack"] != "clean"]
    agg = adv.groupby("model")["accuracy"].mean().reset_index()
    lat_col = f"latency_ms_bs{batch_size}"
    if lat_col not in efficiency_df.columns:
        return None
    merged = agg.merge(efficiency_df[["model", lat_col]], on="model", how="inner").dropna()
    if merged.empty:
        return None
    fig, ax = plt.subplots(figsize=(6.2, 4.2))
    for i, row in merged.iterrows():
        ax.scatter(row[lat_col], row["accuracy"], s=140, color=PUB_PALETTE[i % len(PUB_PALETTE)],
                   edgecolor="black", zorder=3)
        ax.annotate(disp(row["model"]), (row[lat_col], row["accuracy"]),
                     textcoords="offset points", xytext=(8, 4), fontsize=10)
    ax.set_xscale("log")
    ax.set_xlabel(f"Olculen Cikarim Gecikmesi, ms/ornek (batch={batch_size}, log olcek)")
    ax.set_ylabel("Mean robust accuracy (all attacks)")
    ax.set_title("Measured latency versus robustness")
    return _save(fig, f"latency_vs_robust_accuracy_bs{batch_size}")

def plot_defense_effectiveness(defense_df: pd.DataFrame, dataset: str, model: str):
    """Savunmasiz (defense=none) ile FGSM-karisik adversarial-training-ile-
    savunulmus (defense=adv_train) modelin, her saldiri/epsilon kombinasyonunda
    ortalama dogrulugunu YAN YANA karsilastiran gruplu cubuk grafik. Savunmanin
    ISE YARAYIP YARAMADIGINI dogrudan gorsellestirir."""
    set_publication_style()
    sub = defense_df[(defense_df["dataset"] == dataset) & (defense_df["model"] == model)].copy()
    if sub.empty:
        return None
    sub["attack_eps"] = sub.apply(
        lambda r: "Temiz" if r["attack"] == "clean" else f"{r['attack']}\n(eps={r['eps']})", axis=1
    )
    agg = sub.groupby(["attack_eps", "defense"])["accuracy"].mean().reset_index()
    pivot = agg.pivot(index="attack_eps", columns="defense", values="accuracy")
    order = (["Temiz"] if "Temiz" in pivot.index else []) + sorted(
        [i for i in pivot.index if i != "Temiz"]
    )
    pivot = pivot.reindex(order)

    fig, ax = plt.subplots(figsize=(7.5, 4.5))
    x = np.arange(len(pivot))
    width = 0.35
    if "none" in pivot.columns:
        ax.bar(x - width / 2, pivot["none"], width, label="Undefended (standard training)",
               color=PUB_PALETTE[1], edgecolor="black", linewidth=0.6)
    if "adv_train" in pivot.columns:
        ax.bar(x + width / 2, pivot["adv_train"], width, label="FGSM-mixed adversarial training",
               color=PUB_PALETTE[2], edgecolor="black", linewidth=0.6)
    ax.set_xticks(x)
    ax.set_xticklabels(pivot.index, fontsize=9)
    ax.set_ylabel("Ortalama Accuracy")
    ax.set_ylim(0, 1.05)
    ax.set_title(f"{dataset} / {model}: Adversarial Training Savunmasinin Etkinligi")
    ax.legend(frameon=False)
    return _save(fig, f"defense_effectiveness_{dataset}_{model}")

def plot_ablation_sensitivity(ablation_df: pd.DataFrame, dataset: str):
    """Her mimarinin FARKLI d_model degerlerindeki temiz dogrulugunu gosteren
    gruplu cubuk grafik -- 'sonuclar hiperparametre secimine ne kadar duyarli?'
    sorusunun dogrudan gorsel yaniti. Siralamalarin d_model degerleri arasinda
    TUTARLI kalmasi, ana bulgunun keyfi bir hiperparametre secimine bagli
    OLMADIGINI gosterir."""
    set_publication_style()
    sub = ablation_df[ablation_df["dataset"] == dataset]
    if sub.empty:
        return None
    agg = sub.groupby(["model", "d_model"])["accuracy"].mean().reset_index()
    pivot = agg.pivot(index="model", columns="d_model", values="accuracy").sort_index()

    fig, ax = plt.subplots(figsize=(7, 4.3))
    x = np.arange(len(pivot))
    n_bars = len(pivot.columns)
    width = 0.8 / max(n_bars, 1)
    for i, d_model_val in enumerate(pivot.columns):
        offset = (i - (n_bars - 1) / 2) * width
        ax.bar(x + offset, pivot[d_model_val], width, label=f"d_model={d_model_val}",
               color=PUB_PALETTE[i % len(PUB_PALETTE)], edgecolor="black", linewidth=0.6)
    ax.set_xticks(x)
    ax.set_xticklabels(pivot.index)
    ax.set_ylabel("Clean test accuracy")
    ax.set_ylim(0, 1.05)
    ax.set_title(f"{dataset}: Hiperparametre (d_model) Duyarlilik Ablasyonu")
    ax.legend(frameon=False, title="Model capacity")
    return _save(fig, f"ablation_sensitivity_{dataset}")

print("Yayina-hazir grafik modulu hazir (300dpi PNG + vektor PDF, Drive/figures altina kaydedilir).")


## 13. Yayına-Hazır Tablo Üretimi (CSV + LaTeX + Markdown)

In [ ]:
# ============================================================
# HÜCRE 13 — YAYINA-HAZIR TABLO URETIMI (CSV + LaTeX + Markdown)
# ============================================================
import pandas as pd

def save_table(df: pd.DataFrame, name: str, caption: str = "", float_fmt: str = "%.4f"):
    """Bir DataFrame'i UC formatta Drive/tables altina kaydeder:
    1) CSV  (Excel/istatistik yazilimlari icin)
    2) .tex (LaTeX booktabs, dogrudan makaleye \\input{} ile eklenebilir)
    3) .md  (hizli inceleme / on-yazi icin)"""
    csv_path = f"{PATHS['tables']}/{name}.csv"
    tex_path = f"{PATHS['tables']}/{name}.tex"
    md_path = f"{PATHS['tables']}/{name}.md"

    df.to_csv(csv_path, index=False, float_format=float_fmt)

    try:
        latex_body = df.to_latex(index=False, float_format=lambda x: float_fmt % x, escape=True)
    except Exception:
        latex_body = df.to_latex(index=False, escape=True)
    latex_full = (
        "% Otomatik uretildi - BTD makalesi icin yayina-hazir tablo\n"
        "\\begin{table}[ht]\n\\centering\n"
        f"\\caption{{{caption or name}}}\n"
        f"\\label{{tab:{name}}}\n"
        + latex_body.replace("tabular", "tabular").replace("\\toprule", "\\toprule")
        + "\n\\end{table}\n"
    )
    with open(tex_path, "w") as f:
        f.write(latex_full)

    with open(md_path, "w") as f:
        f.write(f"### {caption or name}\n\n")
        f.write(df.to_markdown(index=False, floatfmt=".4f"))

    print(f"[TABLO KAYDEDILDI] {csv_path} | {tex_path} | {md_path}")
    return csv_path, tex_path, md_path

def build_summary_table(results_df: pd.DataFrame) -> pd.DataFrame:
    """Veri seti x model x saldiri kirilimda ORTALAMA +/- STD (coklu tohum)
    ozet tablosu - makalenin ana sonuc tablosu icin."""
    agg = (
        results_df.groupby(["dataset", "model", "attack", "eps"])
        .agg(accuracy_mean=("accuracy", "mean"), accuracy_std=("accuracy", "std"),
             precision_mean=("precision_macro", "mean"),
             precision_std=("precision_macro", "std"),
             recall_mean=("recall_macro", "mean"),
             recall_std=("recall_macro", "std"),
             f1_mean=("f1_macro", "mean"), f1_std=("f1_macro", "std"),
             auc_mean=("auc", "mean"), auc_std=("auc", "std"))
        .reset_index()
        .sort_values(["dataset", "attack", "eps", "accuracy_mean"], ascending=[True, True, True, False])
    )
    return agg

print("Tablo modulu hazir: save_table(df, name, caption), build_summary_table(results_df).")


## 14. Ana Orkestrasyon Pipeline'ı

Üç alt-bölüm içerir:
- **14a** `run_full_pipeline()`: dataset × model × tohum döngüsü — eğitim, saldırı, değerlendirme.
- **14b** `run_adversarial_training_defense()`: Mamba ve en güçlü rakip için FGSM-karışık
  adversarial training savunmasının etkinliğini değerlendirir.
- **14c** `run_hyperparameter_ablation()`: farklı `d_model` değerlerinde küçük çaplı
  bir duyarlılık analizi — ana bulguların keyfi hiperparametre seçimine bağlı
  olmadığını gösterir.


In [ ]:
# ============================================================
# HÜCRE 14 — ANA ORKESTRASYON PIPELINE'I
# dataset x model x seed dongusu: egit -> temiz degerlendir -> saldir ->
# sağlam degerlendir -> sinif-bazli kirilganlik -> tum sonuclari topla.
# Her asama checkpoint'lidir; Colab oturumu koparsa 'Run All' TEKRAR
# calistirildiginda TAMAMLANMIS isler ATLANIR (zaman/GPU-saati tasarrufu).
# ============================================================
import pandas as pd
import numpy as np
import gc

MASTER_RESULTS_PATH = f"{PATHS['results']}/master_results_{RUN_ID}.csv"
PARAM_COUNTS_PATH = f"{PATHS['results']}/param_counts_{RUN_ID}.json"

def count_params(model) -> int:
    if hasattr(model, "parameters"):
        return sum(p.numel() for p in model.parameters())
    return -1  # RandomForest icin tanimsiz (agac-tabanli, "parametre" kavrami farkli)

def run_full_pipeline():
    all_rows = []
    param_counts = {}
    total_jobs = len(CONFIG["datasets"]) * len(CONFIG["models"]) * len(CONFIG["seeds"])
    job_idx = 0

    for dataset in CONFIG["datasets"]:
        data = DATA_CACHE[dataset]
        X_train, y_train = data["X_train"], data["y_train"]
        X_test, y_test = data["X_test"], data["y_test"]
        clip_min, clip_max = get_feature_clip_bounds(X_train)

        # Kara-kutu saldirisi icin sabit bir vekil (surrogate) mimari secilir
        # (hedeften FARKLI bir aile: CNN1D -> her hedef icin ayri egitilir).
        surrogate_cache = {}

        for model_name in CONFIG["models"]:
            seed_metrics = {"clean": [], **{f"{a}_{e}": [] for a in CONFIG["attacks"] for e in [0]}}
            for seed in CONFIG["seeds"]:
                job_idx += 1
                stage_tag = f"{dataset}/{model_name}/seed{seed} ({job_idx}/{total_jobs})"
                print(f"\n{'='*70}\n[ASAMA {job_idx}/{total_jobs}] {stage_tag}\n{'='*70}")

                with EmailOnError(f"Egitim: {stage_tag}"):
                    model, meta = train_model(
                        model_name, dataset, seed, X_train, y_train, X_test, y_test, verbose=True
                    )
                param_counts[model_name] = count_params(model)

                # ---- Temiz (clean) degerlendirme ----
                probs_clean = get_probs(model, X_test, DEVICE, model_name)
                m_clean = compute_metrics(y_test.numpy(), probs_clean)
                all_rows.append({"dataset": dataset, "model": model_name, "seed": seed,
                                  "attack": "clean", "eps": 0.0, **m_clean})

                # RandomForest'a gradyan-tabanli saldiri UYGULANAMAZ (turevlenebilir degil);
                # literatur ile tutarli olarak sadece kara-kutu transfer saldirisi uygulanir.
                if model_name != "RandomForest":
                    with EmailOnError(f"Beyaz-kutu saldirilar: {stage_tag}"):
                        for attack_name, atk_cfg in CONFIG["attacks"].items():
                            if attack_name == "BlackBoxTransfer":
                                continue
                            for eps in atk_cfg["eps_list"]:
                                X_adv = generate_adversarial_examples(
                                    model, X_test, y_test, attack_name, eps, clip_min, clip_max,
                                    DEVICE, pgd_alpha=atk_cfg.get("alpha"), pgd_steps=atk_cfg.get("steps", 10),
                                )
                                probs_adv = get_probs(model, X_adv, DEVICE, model_name)
                                m_adv = compute_metrics(y_test.numpy(), probs_adv)
                                all_rows.append({"dataset": dataset, "model": model_name, "seed": seed,
                                                  "attack": attack_name, "eps": eps, **m_adv})

                    # ---- Kara-kutu transfer saldirisi ----
                    with EmailOnError(f"Kara-kutu transfer saldirisi: {stage_tag}"):
                        surrogate_key = f"{dataset}_surrogate_seed{seed}"
                        if surrogate_key not in surrogate_cache:
                            surrogate_arch = "LSTM" if model_name != "LSTM" else "CNN1D"
                            surrogate_model, _ = train_model(
                                surrogate_arch, dataset, seed, X_train, y_train, X_test, y_test, verbose=False
                            )
                            surrogate_cache[surrogate_key] = surrogate_model
                        surrogate_model = surrogate_cache[surrogate_key]

                        for eps in CONFIG["attacks"]["BlackBoxTransfer"]["eps_list"]:
                            bb = blackbox_transfer_attack(
                                surrogate_model, model, X_test, y_test, eps, clip_min, clip_max, DEVICE
                            )
                            all_rows.append({"dataset": dataset, "model": model_name, "seed": seed,
                                              "attack": "BlackBoxTransfer", "eps": eps,
                                              "accuracy": bb["adv_acc"], "precision_macro": np.nan,
                                              "recall_macro": np.nan, "f1_macro": np.nan, "auc": np.nan})

                # Bellek temizligi (A100 uzun kosularda VRAM sizintisini onler)
                del model
                gc.collect()
                torch.cuda.empty_cache()

            # Ara-kayit: her (dataset, model) tamamlandiginda diske yaz (dayaniklilik)
            pd.DataFrame(all_rows).to_csv(MASTER_RESULTS_PATH, index=False)
            print(f"[ARA-KAYIT] {MASTER_RESULTS_PATH} guncellendi ({len(all_rows)} satir).")

    results_df = pd.DataFrame(all_rows)
    results_df.to_csv(MASTER_RESULTS_PATH, index=False)
    with open(PARAM_COUNTS_PATH, "w") as f:
        json.dump(param_counts, f, indent=2)
    print(f"\n[TAMAMLANDI] Toplam {len(results_df)} sonuc satiri -> {MASTER_RESULTS_PATH}")
    return results_df, param_counts

print("Ana orkestrasyon fonksiyonu hazir: run_full_pipeline()")


# ============================================================
# 14b — ADVERSARIAL TRAINING SAVUNMASI DEGERLENDIRMESI
# Sadece OLCMEKLE kalmayip, bulgulara dayali bir IYILESTIRME de sunmak icin:
# secili mimariler (varsayilan: Mamba ve CNN1D) FGSM-karisik adversarial
# training ile YENIDEN egitilir ve savunmasiz (standart-egitilmis) haliyle
# robust dogruluk acisindan KARSILASTIRILIR.
# ============================================================
DEFENSE_RESULTS_PATH = f"{PATHS['results']}/defense_results_{RUN_ID}.csv"

def run_adversarial_training_defense(results_df: pd.DataFrame) -> pd.DataFrame:
    adv_cfg = CONFIG.get("adversarial_training", {})
    if not adv_cfg.get("enabled", False):
        print("[BILGI] Adversarial training savunmasi devre disi (CONFIG['adversarial_training']['enabled']=False).")
        return pd.DataFrame()

    defense_rows = []
    target_models = [m for m in adv_cfg.get("models", []) if m != "RandomForest"]

    # ---- (a) 'defense=none' referans satirlari: zaten run_full_pipeline()
    # tarafindan hesaplanmis standart sonuclardan devsirilir (TEKRAR EGITIM YOK) ----
    baseline = results_df[results_df["model"].isin(target_models)].copy()
    baseline["defense"] = "none"
    defense_rows.append(baseline)

    # ---- (b) 'defense=adv_train': FGSM-karisik adversarial training ile YENIDEN egitim ----
    for dataset in CONFIG["datasets"]:
        data = DATA_CACHE[dataset]
        X_train, y_train = data["X_train"], data["y_train"]
        X_test, y_test = data["X_test"], data["y_test"]
        clip_min, clip_max = get_feature_clip_bounds(X_train)

        for model_name in target_models:
            for seed in CONFIG["seeds"]:
                stage_tag = f"[ADV-TRAIN] {dataset}/{model_name}/seed{seed}"
                print(f"\n{stage_tag}")
                with EmailOnError(f"Adversarial training: {stage_tag}"):
                    model, _ = train_model(
                        model_name, dataset, seed, X_train, y_train, X_test, y_test,
                        verbose=True, adv_training=True,
                        adv_eps=adv_cfg.get("eps", 0.05), adv_mix_ratio=adv_cfg.get("mix_ratio", 0.5),
                    )

                probs_clean = get_probs(model, X_test, DEVICE, model_name)
                m_clean = compute_metrics(y_test.numpy(), probs_clean)
                defense_rows.append(pd.DataFrame([{
                    "dataset": dataset, "model": model_name, "seed": seed, "defense": "adv_train",
                    "attack": "clean", "eps": 0.0, **m_clean,
                }]))

                for attack_name, atk_cfg in CONFIG["attacks"].items():
                    if attack_name == "BlackBoxTransfer":
                        continue
                    for eps in atk_cfg["eps_list"]:
                        X_adv = generate_adversarial_examples(
                            model, X_test, y_test, attack_name, eps, clip_min, clip_max, DEVICE,
                            pgd_alpha=atk_cfg.get("alpha"), pgd_steps=atk_cfg.get("steps", 10),
                        )
                        probs_adv = get_probs(model, X_adv, DEVICE, model_name)
                        m_adv = compute_metrics(y_test.numpy(), probs_adv)
                        defense_rows.append(pd.DataFrame([{
                            "dataset": dataset, "model": model_name, "seed": seed, "defense": "adv_train",
                            "attack": attack_name, "eps": eps, **m_adv,
                        }]))

                del model
                gc.collect()
                torch.cuda.empty_cache()

    defense_df = pd.concat(defense_rows, ignore_index=True)
    defense_df.to_csv(DEFENSE_RESULTS_PATH, index=False)
    print(f"\n[TAMAMLANDI] Savunma degerlendirmesi -> {DEFENSE_RESULTS_PATH} ({len(defense_df)} satir)")
    return defense_df

print("Savunma degerlendirme fonksiyonu hazir: run_adversarial_training_defense(results_df)")


# ============================================================
# 14c — HIPERPARAMETRE ADALETI ABLASYONU
# GEREKCE: Ana deneyde TUM mimariler ayni (d_model, n_layers) degerlerini
# kullanir (adil karsilastirma icin). Ancak bir hakem haklı olarak sorabilir:
# "Bu sabit secim, bilhassa bir mimariyi mi kayiriyor?" Bu fonksiyon, her
# mimariyi FARKLI d_model degerlerinde (kucuk hesaplama yukuyle, az sayida
# tohum/epoch ile) egiterek, ana bulgularin (Mamba'nin goreli sirasi, vb.)
# hiperparametre secimine DUYARLI olup olmadigini gosterir.
# ============================================================
ABLATION_RESULTS_PATH = f"{PATHS['results']}/ablation_results_{RUN_ID}.csv"

def run_hyperparameter_ablation() -> pd.DataFrame:
    abl_cfg = CONFIG.get("ablation", {})
    if not abl_cfg.get("enabled", False):
        print("[BILGI] Hiperparametre ablasyonu devre disi (CONFIG['ablation']['enabled']=False).")
        return pd.DataFrame()

    d_model_values = abl_cfg.get("d_model_values", [CONFIG["d_model"]])
    abl_seeds = abl_cfg.get("seeds", CONFIG["seeds"][:2])
    torch_models = [m for m in CONFIG["models"] if m != "RandomForest"]

    abl_rows = []
    for dataset in CONFIG["datasets"]:
        data = DATA_CACHE[dataset]
        X_train, y_train = data["X_train"], data["y_train"]
        X_test, y_test = data["X_test"], data["y_test"]

        for model_name in torch_models:
            for d_model in d_model_values:
                for seed in abl_seeds:
                    stage_tag = f"[ABLASYON] {dataset}/{model_name}/d_model={d_model}/seed{seed}"
                    print(f"\n{stage_tag}")
                    # Gecici olarak CONFIG['d_model'] degistirilir (build_model bunu okur);
                    # islem sonunda ORIJINAL deger GERI YUKLENIR (yan etki sizintisi onlenir).
                    original_d_model = CONFIG["d_model"]
                    CONFIG["d_model"] = d_model
                    try:
                        with EmailOnError(f"Ablasyon: {stage_tag}"):
                            model, meta = train_model(
                                model_name, f"{dataset}_ablation_dm{d_model}", seed,
                                X_train, y_train, X_test, y_test, verbose=False,
                            )
                        probs_clean = get_probs(model, X_test, DEVICE, model_name)
                        m_clean = compute_metrics(y_test.numpy(), probs_clean)
                        n_params = sum(p.numel() for p in model.parameters())
                        abl_rows.append({
                            "dataset": dataset, "model": model_name, "d_model": d_model, "seed": seed,
                            "n_parameters": n_params, **m_clean,
                        })
                        del model
                        torch.cuda.empty_cache()
                    finally:
                        CONFIG["d_model"] = original_d_model

    if not abl_rows:
        return pd.DataFrame()
    abl_df = pd.DataFrame(abl_rows)
    abl_df.to_csv(ABLATION_RESULTS_PATH, index=False)
    print(f"\n[TAMAMLANDI] Hiperparametre ablasyonu -> {ABLATION_RESULTS_PATH} ({len(abl_df)} satir)")
    return abl_df

print("Ablasyon fonksiyonu hazir: run_hyperparameter_ablation()")


## 15. Tüm Grafik/Tablo Üretimi + İstatistiksel Anlamlılık Testleri

In [ ]:
# ============================================================
# HÜCRE 15 — TUM GRAFIK/TABLO URETIMI + ISTATISTIKSEL ANLAMLILIK TESTLERI
# Bu hucre, run_full_pipeline() TAMAMLANDIKTAN SONRA calisir ve makalenin
# "Bulgular" bolumune dogrudan konulabilecek TUM cikti materyalini uretir.
# ============================================================
def generate_all_outputs(results_df: pd.DataFrame, param_counts: dict, defense_df: pd.DataFrame = None,
                          ablation_df: pd.DataFrame = None):
    generated_figures, generated_tables = [], []
    efficiency_df = None  # Hucre 16'da dinamik verimlilik anlatisi icin disari aktarilir

    with EmailOnError("Yayina-hazir cikti uretimi (grafik + tablo)"):
        # ---- 1) Ana ozet tablosu (dataset x model x saldiri x eps) ----
        summary = build_summary_table(results_df)
        generated_tables.append(save_table(
            summary, "T1_ana_ozet_sonuclari",
            caption="Tum veri setleri, modeller ve saldirilar icin ortalama +/- standart sapma "
                    "(3 rastgele tohum uzerinden) test dogrulugu ve F1-skoru.",
        ))

        # ---- 2) Veri seti basina grafikler ----
        for dataset in results_df["dataset"].unique():
            fig = plot_clean_accuracy_bar(results_df, dataset)
            if fig: generated_figures.append(fig)
            for attack in ["FGSM", "PGD"]:
                fig = plot_robustness_curve(results_df, dataset, attack)
                if fig: generated_figures.append(fig)

        # ---- 3) (Verimlilik-sağlamlik grafikleri Bolum 6'da, GERCEK OLCULEN
        # gecikme verisiyle birlikte uretilir -- bkz. asagisi) ----

        # ---- 4) Istatistiksel anlamlilik: Mamba vs. en iyi baseline (her veri seti x saldiri) ----
        # ============================================================
        # REVIZYON (secim yanliligi): Karsilastirma modelini "en iyi rakip"
        # olarak SONRADAN secmek, testte kullanilan AYNI veriye bakarak
        # maksimumu secmek demektir; Holm duzeltmesi hucreler ARASI coklugu
        # duzeltir ama hucre ICINDEKI bu secimi DUZELTMEZ. Bu nedenle:
        #   BIRINCIL  : karsilastirma modeli ONCEDEN Transformer olarak sabit
        #   IKINCIL   : eski (post-hoc en iyi rakip) analiz, seffaflik icin
        # ============================================================
        PRESPECIFIED_COMPARATOR = "Transformer"
        sig_rows, sig_rows_posthoc = [], []
        for dataset in results_df["dataset"].unique():
            for attack in results_df["attack"].unique():
                sub = results_df[(results_df["dataset"] == dataset) & (results_df["attack"] == attack)]
                if "Mamba" not in sub["model"].unique():
                    continue
                mamba_scores = sub[sub["model"] == "Mamba"].groupby("seed")["accuracy"].mean()

                # ---- BIRINCIL: onceden belirlenmis karsilastirma modeli ----
                if PRESPECIFIED_COMPARATOR in sub["model"].unique():
                    pre_scores = sub[sub["model"] == PRESPECIFIED_COMPARATOR].groupby("seed")["accuracy"].mean()
                    ks = sorted(set(mamba_scores.index) & set(pre_scores.index))
                    if len(ks) >= 2:
                        res_pre = paired_significance_test(
                            mamba_scores.loc[ks].tolist(), pre_scores.loc[ks].tolist())
                        sig_rows.append({"dataset": dataset, "attack": attack,
                                         "karsilastirma_modeli": PRESPECIFIED_COMPARATOR,
                                         "mamba_ortalama": mamba_scores.loc[ks].mean(),
                                         "rakip_ortalama": pre_scores.loc[ks].mean(),
                                         **res_pre})

                # ---- IKINCIL: post-hoc en iyi rakip (anti-konservatif) ----
                other_models = [m for m in sub["model"].unique() if m != "Mamba"]
                if not other_models:
                    continue
                best_other = sub[sub["model"] != "Mamba"].groupby("model")["accuracy"].mean().idxmax()
                other_scores = sub[sub["model"] == best_other].groupby("seed")["accuracy"].mean()
                common_seeds = sorted(set(mamba_scores.index) & set(other_scores.index))
                if len(common_seeds) < 2:
                    continue
                test_res = paired_significance_test(
                    mamba_scores.loc[common_seeds].tolist(), other_scores.loc[common_seeds].tolist()
                )
                sig_rows_posthoc.append({"dataset": dataset, "attack": attack,
                                          "en_iyi_karsilastirma_modeli": best_other,
                                          "mamba_ortalama": mamba_scores.mean(),
                                          "rakip_ortalama": other_scores.mean(),
                                          **test_res})
        # ---- IKINCIL tabloyu da kaydet ----
        if sig_rows_posthoc:
            ph_df = pd.DataFrame(sig_rows_posthoc)
            ph_df["p_value_holm"] = holm_bonferroni_correction(ph_df["p_value"].tolist())
            generated_tables.append(save_table(
                ph_df, "T2b_istatistiksel_anlamlilik_posthoc",
                caption="IKINCIL analiz: her hucrede EMPIRIK olarak en guclu rakiple "
                        "karsilastirma. Karsilastirma modeli, testte kullanilan AYNI "
                        "veriye bakilarak secildiginden p-degerleri ANTI-KONSERVATIFTIR "
                        "ve yalnizca seffaflik icin raporlanir; Holm duzeltmesi bu secim "
                        "etkisini gidermez. Cikarim icin T2 (birincil) tablosu kullanilmalidir.",
            ))

        if sig_rows:
            sig_df = pd.DataFrame(sig_rows)
            # ONEMLI (v3 revizyonu): coklu (veri-seti x saldiri) karsilastirmasi yapildigindan
            # Holm-Bonferroni duzeltmesi UYGULANIR (aile-capinda Tip-I hata kontrolu). Hem HAM
            # (p_value) hem DUZELTILMIS (p_value_holm) sutunlar birlikte raporlanir -- seffaflik icin.
            sig_df["p_value_holm"] = holm_bonferroni_correction(sig_df["p_value"].tolist())
            sig_df["anlamli_ham_p<0.05"] = sig_df["p_value"].apply(lambda p: p is not None and p < 0.05)
            sig_df["anlamli_holm_p<0.05"] = sig_df["p_value_holm"].apply(lambda p: p is not None and p < 0.05)
            generated_tables.append(save_table(
                sig_df, "T2_istatistiksel_anlamlilik",
                caption="BIRINCIL analiz: Mamba ile ONCEDEN BELIRLENMIS karsilastirma modeli "
                        "(Transformer) arasinda esli t-testi ve Wilcoxon isaretli-sira "
                        "testi sonuclari. p_value: HAM (duzeltilmemis) p-degeri; p_value_holm: coklu-"
                        "karsilastirma icin Holm-Bonferroni ile DUZELTILMIS p-degeri (aile-capinda hata "
                        "oranini kontrol eder). Anlamlilik yorumlanirken p_value_holm sutunu ESAS ALINMALIDIR.",
            ))

        # ---- 5) Sinif-bazli VE saldiri-alt-turu-bazli kirilganlik (en yuksek eps, FGSM) ----
        # NOT: Bu bolum icin modelin tekrar test setinde calistirilmasi gerekir;
        # run_full_pipeline() sirasinda checkpoint'ler kaydedildigi icin modeller
        # Drive'dan yeniden yuklenerek tekrar EGITILMEDEN kullanilabilir.
        category_vuln_rows = []  # T3 ozet tablosu icin (tum dataset x model kirilimi)
        for dataset in CONFIG["datasets"]:
            data = DATA_CACHE[dataset]
            clip_min, clip_max = get_feature_clip_bounds(data["X_train"])
            max_eps = max(CONFIG["attacks"]["FGSM"]["eps_list"])
            for model_name in [m for m in CONFIG["models"] if m != "RandomForest"]:
                ckpt_p = checkpoint_path(dataset, model_name, CONFIG["seeds"][0])
                if not os.path.exists(ckpt_p):
                    continue
                model = build_model(model_name, CONFIG["seq_len"], CONFIG["d_model"], CONFIG["n_layers"]).to(DEVICE)
                model.load_state_dict(torch.load(ckpt_p, map_location=DEVICE, weights_only=False)["model_state"])
                model.eval()
                probs_clean = get_probs(model, data["X_test"], DEVICE, model_name)
                X_adv = generate_adversarial_examples(
                    model, data["X_test"], data["y_test"], "FGSM", max_eps, clip_min, clip_max, DEVICE
                )
                probs_adv = get_probs(model, X_adv, DEVICE, model_name)

                # 5a) Ikili (benign/saldiri) sinif-bazli kirilganlik
                vuln_df = compute_per_class_vulnerability(data["y_test"].numpy(), probs_clean, probs_adv)
                fig = plot_per_class_heatmap(vuln_df, dataset, model_name, f"FGSM(eps={max_eps})")
                if fig: generated_figures.append(fig)

                # 5b) SALDIRI ALT-TURU bazli kirilganlik (asil arastirma sorusu: "hangi
                # saldiri turu Mamba'yi/rakiplerini en cok kandiriyor?")
                cat_vuln_df = compute_attack_category_vulnerability(
                    data["y_test"].numpy(), data["attack_category_test"], probs_clean, probs_adv
                )
                fig = plot_attack_category_vulnerability(cat_vuln_df, dataset, model_name, f"FGSM(eps={max_eps})")
                if fig: generated_figures.append(fig)
                if not cat_vuln_df.empty:
                    cat_vuln_df = cat_vuln_df.copy()
                    cat_vuln_df["dataset"] = dataset
                    cat_vuln_df["model"] = model_name
                    category_vuln_rows.append(cat_vuln_df)

                del model
                torch.cuda.empty_cache()

        if category_vuln_rows:
            all_cat_vuln = pd.concat(category_vuln_rows, ignore_index=True)
            all_cat_vuln = all_cat_vuln[
                ["dataset", "model", "attack_category", "n_samples",
                 "clean_detection_rate", "adv_detection_rate", "evasion_success_rate"]
            ].sort_values(["dataset", "evasion_success_rate"], ascending=[True, False])
            generated_tables.append(save_table(
                all_cat_vuln, "T3_saldiri_turu_kirilganligi",
                caption=f"FGSM (epsilon={max(CONFIG['attacks']['FGSM']['eps_list'])}) saldirisi altinda, "
                        "her veri seti ve model icin saldiri ALT-TURU bazinda atlatma (evasion) basari "
                        "oranlari. Yuksek oran, o saldiri kategorisinin ilgili mimari tarafindan kolayca "
                        "'goruniyor -> benign' seklinde yanlis siniflandirildigini gosterir.",
            ))

        # ---- 6) CIKARIM HIZI (latency/throughput) OLCUMU ----
        # Parametre sayisindan daha ikna edici bir verimlilik kaniti: GERCEK
        # OLCULEN gecikme (bkz. Hucre 9b). Ilk tohumdaki egitilmis checkpoint
        # kullanilir (agirliklar gecikmeyi ETKILEMEZ, sadece mimari onemlidir,
        # ama tutarlilik icin ayni checkpoint'ler tercih edilir).
        latency_results = {}
        for dataset in CONFIG["datasets"][:1]:  # gecikme mimariye bagli, veri setinden BAGIMSIZDIR -> bir kez olcmek yeterli
            data = DATA_CACHE[dataset]
            for model_name in CONFIG["models"]:
                if model_name == "RandomForest":
                    ckpt_p = None
                    rf_model, _ = train_model(model_name, dataset, CONFIG["seeds"][0],
                                               data["X_train"], data["y_train"], data["X_test"], data["y_test"],
                                               verbose=False)
                    latency_results[model_name] = measure_inference_latency(
                        rf_model, CONFIG["seq_len"], DEVICE, model_name, batch_sizes=(1, 512)
                    )
                    continue
                ckpt_p = checkpoint_path(dataset, model_name, CONFIG["seeds"][0])
                if not os.path.exists(ckpt_p):
                    continue
                model = build_model(model_name, CONFIG["seq_len"], CONFIG["d_model"], CONFIG["n_layers"]).to(DEVICE)
                model.load_state_dict(torch.load(ckpt_p, map_location=DEVICE, weights_only=False)["model_state"])
                latency_results[model_name] = measure_inference_latency(
                    model, CONFIG["seq_len"], DEVICE, model_name, batch_sizes=(1, 512)
                )
                del model
                torch.cuda.empty_cache()

        if latency_results:
            efficiency_df = build_efficiency_table(param_counts, latency_results, batch_sizes=(1, 512))
            generated_tables.append(save_table(
                efficiency_df, "T4_cikarim_verimliligi",
                caption="Model basina parametre sayisi, tek-ornek (batch=1, gercek-zamanli senaryo) ve "
                        "toplu-isleme (batch=512) cikarim gecikmesi (ms/ornek) ve verimi (ornek/saniye). "
                        "GPU olcumleri torch.cuda.Event ile senkronize edilerek yapilmistir.",
            ))
            for bs in (1, 512):
                fig = plot_latency_vs_robust_accuracy(results_df, efficiency_df, batch_size=bs)
                if fig: generated_figures.append(fig)
            fig = plot_params_vs_robust_accuracy(results_df, param_counts)
            if fig: generated_figures.append(fig)

        # ---- 7) ADVERSARIAL TRAINING SAVUNMASININ ETKINLIGI ----
        if defense_df is not None and not defense_df.empty:
            defense_summary = (
                defense_df.groupby(["dataset", "model", "defense", "attack", "eps"])
                .agg(accuracy_mean=("accuracy", "mean"), accuracy_std=("accuracy", "std"))
                .reset_index()
            )
            generated_tables.append(save_table(
                defense_summary, "T5_savunma_etkinligi",
                caption="FGSM-karisik adversarial training savunmasi UYGULANMADAN once (defense=none) "
                        "ve SONRA (defense=adv_train) elde edilen ortalama dogruluklarin karsilastirmasi. "
                        "adv_train sutunundaki degerlerin none sutunundan YUKSEK olmasi, savunmanin "
                        "etkili oldugunu gosterir.",
            ))
            for dataset in defense_df["dataset"].unique():
                for model in defense_df[defense_df["dataset"] == dataset]["model"].unique():
                    fig = plot_defense_effectiveness(defense_df, dataset, model)
                    if fig: generated_figures.append(fig)

        # ---- 8) HIPERPARAMETRE ADALETI ABLASYONU ----
        if ablation_df is not None and not ablation_df.empty:
            ablation_summary = (
                ablation_df.groupby(["dataset", "model", "d_model"])
                .agg(accuracy_mean=("accuracy", "mean"), accuracy_std=("accuracy", "std"),
                     n_parameters=("n_parameters", "first"))
                .reset_index()
                .sort_values(["dataset", "model", "d_model"])
            )
            generated_tables.append(save_table(
                ablation_summary, "T6_hiperparametre_ablasyonu",
                caption="Her mimarinin FARKLI kapasite (d_model) degerlerindeki temiz test "
                        "dogrulugu. Modeller-arasi SIRALAMANIN d_model degerleri boyunca "
                        "TUTARLI kalmasi, ana deneydeki sabit hiperparametre seciminin "
                        "sonuclari onyargilamadigini destekler.",
            ))
            for dataset in ablation_df["dataset"].unique():
                fig = plot_ablation_sensitivity(ablation_df, dataset)
                if fig: generated_figures.append(fig)

    print(f"\nToplam {len(generated_figures)} grafik ve {len(generated_tables)} tablo uretildi.")
    print(f"Grafikler -> {PATHS['figures']}")
    print(f"Tablolar  -> {PATHS['tables']}")
    return generated_figures, generated_tables, efficiency_df

print("Rapor uretim fonksiyonu hazir: generate_all_outputs(results_df, param_counts)")


## 16. Final Özet Raporu (Markdown)

In [ ]:
# ============================================================
# HÜCRE 16 — FINAL OZET RAPORU (Markdown) URETIMI
# ============================================================
import datetime

def generate_final_report(results_df: pd.DataFrame, param_counts: dict,
                           defense_df: pd.DataFrame = None, ablation_df: pd.DataFrame = None,
                           efficiency_df: pd.DataFrame = None) -> str:
    report_path = f"{PATHS['reports']}/final_report_{RUN_ID}.md"
    now = datetime.datetime.now(datetime.timezone.utc).strftime("%Y-%m-%d %H:%M UTC")

    clean = results_df[results_df["attack"] == "clean"]
    best_clean = (
        clean.groupby("model")["accuracy"].mean().sort_values(ascending=False)
        if not clean.empty else pd.Series(dtype=float)
    )

    adv = results_df[results_df["attack"] != "clean"]
    most_robust = (
        adv.groupby("model")["accuracy"].mean().sort_values(ascending=False)
        if not adv.empty else pd.Series(dtype=float)
    )

    lines = [
        f"# BTD Makalesi — Deneysel Sonuc Ozeti",
        f"**Calisma Kimligi (Run ID):** {RUN_ID}  ",
        f"**Olusturulma zamani:** {now}  ",
        f"**GPU:** {gpu_name if 'gpu_name' in globals() else 'bilinmiyor'}  ",
        "",
        "## 1. Kapsam",
        f"- Veri setleri: {', '.join(CONFIG['datasets'])}",
        f"- Modeller: {', '.join(CONFIG['models'])}",
        f"- Tohumlar (istatistiksel tekrar): {CONFIG['seeds']} (n={len(CONFIG['seeds'])})",
        f"- Saldirilar: FGSM {CONFIG['attacks']['FGSM']['eps_list']}, "
        f"PGD {CONFIG['attacks']['PGD']['eps_list']}, "
        f"Black-Box Transfer {CONFIG['attacks']['BlackBoxTransfer']['eps_list']}",
        f"- Toplam sonuc satiri: {len(results_df)}",
        "",
        "## 2. Model Parametre Sayilari",
        "| Model | Parametre Sayisi |",
        "|---|---|",
    ]
    for m, c in param_counts.items():
        lines.append(f"| {m} | {c if c > 0 else 'N/A (agac-tabanli)'} |")

    lines += ["", "## 3. Temiz Veride En Iyi Performans (ortalama dogruluk)"]
    for m, v in best_clean.items():
        lines.append(f"- **{m}**: {v:.4f}")

    lines += ["", "## 4. Saldiri-Alti (Robust) Ortalama Dogruluk Siralamasi"]
    for m, v in most_robust.items():
        lines.append(f"- **{m}**: {v:.4f}")

    if efficiency_df is not None and not efficiency_df.empty:
        lines += ["", "## 5. Ölçülen Çıkarım Verimliliği — Bulgu (OTOMATİK ÜRETİLDİ, yön ne olursa olsun)"]
        for bs, bs_label in [(1, "tek-örnek / gerçek-zamanlı senaryo"), (512, "toplu-işleme senaryosu")]:
            lat_col = f"latency_ms_bs{bs}"
            if lat_col not in efficiency_df.columns:
                continue
            valid = efficiency_df.dropna(subset=[lat_col]).sort_values(lat_col)
            if valid.empty:
                continue
            fastest = valid.iloc[0]
            slowest = valid.iloc[-1]
            mamba_row = valid[valid["model"] == "Mamba"]
            lines.append(f"- **Batch={bs} ({bs_label})**: en hızlı model **{fastest['model']}** "
                         f"({fastest[lat_col]:.4f} ms/örnek), en yavaş model **{slowest['model']}** "
                         f"({slowest[lat_col]:.4f} ms/örnek).")
            if not mamba_row.empty:
                mamba_lat = mamba_row.iloc[0][lat_col]
                mamba_rank = int((valid[lat_col] < mamba_lat).sum()) + 1
                if mamba_row.iloc[0]["model"] == fastest["model"]:
                    verdict = ("Mamba bu senaryoda EN HIZLI mimaridir -- doğrusal-karmaşıklık "
                               "iddiasını bu deneysel kurulumda DOĞRULAR.")
                elif mamba_rank <= len(valid) // 2:
                    verdict = ("Mamba ortalamanın ÜSTÜNDE bir hızda, ancak en hızlı değildir -- "
                               "verimlilik avantajı bu SAF-PYTORCH yeniden uygulamada TAM olarak "
                               "gözlemlenmemiştir (bkz. Bölüm 7 metodolojik not).")
                else:
                    verdict = ("Mamba bu senaryoda ORTALAMANIN ALTINDA bir hızdadır -- bu, resmi "
                               "'mamba-ssm' CUDA çekirdekleri yerine kullanılan saf-PyTorch "
                               "Python-döngülü yeniden uygulamanın (bkz. Hücre 9b) beklenen bir "
                               "sonucu olabilir; DOĞRULUK-SAĞLAMLIK bulguları bu durumda dahi "
                               "geçerliliğini KORUR, sadece 'verimlilik' iddiası bu spesifik "
                               "uygulama için sınırlı kalır.")
                lines.append(f"  - Mamba sırası: {mamba_rank}/{len(valid)} (gecikme={mamba_lat:.4f} ms/örnek). {verdict}")

    lines += ["", "## 6. Doğruluk-Sağlamlık-Hız Üçlü Dengesi (Sentez)"]
    if not most_robust.empty:
        top_robust_model = most_robust.index[0]
        lines.append(
            f"- En sağlam (robust) model **{top_robust_model}** olarak ölçülmüştür. Makale metni, "
            f"verimlilik bulgusunun yönüne göre şu şekillerden biriyle çerçevelenebilir:\n"
            f"  - Eğer Mamba hem en sağlam HEM en hızlı ise: 'Mamba doğruluk-sağlamlık-hız "
            f"üçlüsünde baskın bir çözümdür.'\n"
            f"  - Eğer Mamba en sağlam ama en hızlı DEĞİLSE: 'Mamba, hız-sağlamlık dengesinde "
            f"bir ödünleşim (trade-off) sunar -- pratikte kullanım senaryosuna göre CNN/LSTM "
            f"gibi daha hızlı ancak daha az sağlam alternatiflerle karşılaştırılmalıdır.'"
        )

    if defense_df is not None and not defense_df.empty:
        lines += ["", "## 7. Adversarial Training Savunmasinin Etkinligi (Ozet)"]
        for model in defense_df["model"].unique():
            sub = defense_df[(defense_df["model"] == model) & (defense_df["attack"] != "clean")]
            if sub.empty:
                continue
            none_acc = sub[sub["defense"] == "none"]["accuracy"].mean()
            adv_acc = sub[sub["defense"] == "adv_train"]["accuracy"].mean()
            delta = adv_acc - none_acc if pd.notna(adv_acc) and pd.notna(none_acc) else float("nan")
            lines.append(
                f"- **{model}**: savunmasiz ortalama sağlam dogruluk={none_acc:.4f} -> "
                f"adversarial-training-sonrasi={adv_acc:.4f} "
                f"(**{'+' if delta >= 0 else ''}{delta:.4f}**, {'iyilesme' if delta > 0 else 'kotulesme/degisim yok'})"
            )

    lines += [
        "", "## 8. Uretilen Dosyalar",
        f"- Grafikler: `{PATHS['figures']}` (PNG 300dpi + PDF vektor)",
        f"- Tablolar: `{PATHS['tables']}` (CSV + LaTeX + Markdown), ozellikle:",
        "  - `T1_ana_ozet_sonuclari`: tum modeller/saldirilar icin ana dogruluk tablosu",
        "  - `T2_istatistiksel_anlamlilik`: Mamba vs. en iyi rakip, esli t-testi + Wilcoxon",
        "  - `T3_saldiri_turu_kirilganligi`: saldiri ALT-TURU bazinda atlatma basari oranlari",
        "  - `T4_cikarim_verimliligi`: OLCULEN gecikme/verim (parametre sayisi degil)",
        "  - `T5_savunma_etkinligi`: adversarial training oncesi/sonrasi karsilastirma",
        "  - `T6_hiperparametre_ablasyonu`: farkli d_model degerlerinde duyarlilik analizi (varsa)",
        f"- Ham sonuclar: `{MASTER_RESULTS_PATH}`",
        f"- Model checkpoint'leri: `{PATHS['checkpoints']}`",
        "",
        "## 9. Veri Bolme (Split) Stratejisi Seffafligi (v3 revizyonu)",
        "Her veri seti icin FIILEN kullanilan train/test bolme stratejisi (oncelik: "
        "grup-bazli > zaman-bazli > stratified rastgele) asagida raporlanir; bu, akis/host-"
        "duzeyinde olasi veri sizintisi riskinin makale metninde SEFFAF sekilde tartisilmasini saglar.",
        "- **NSL-KDD**: resmi onceden-bolunmus train/test dosyalari (KDDTrain+/KDDTest+) kullanildi -- sizinti riski yok.",
    ] + [
        f"- **{ds}**: {SPLIT_STRATEGY_LOG[ds]}" for ds in CONFIG["datasets"]
        if ds != "NSL-KDD" and ds in SPLIT_STRATEGY_LOG
    ] + [
        "",
        "## 10. Onemli Metodolojik Notlar (Sinirlamalar Dahil)",
        "- Coklu (veri-seti x saldiri) istatistiksel karsilastirma nedeniyle T2 tablosunda "
        "Holm-Bonferroni ile DUZELTILMIS p-degerleri (p_value_holm) de raporlanir; anlamlilik "
        "yorumu HAM p-degeri yerine bu sutuna dayanmalidir (bkz. Hucre 11/15).",
        "- Mamba bloklari, ozel CUDA cekirdegi gerektirmeyen, saf PyTorch S6 "
        "(secici durum-uzayi) formulasyonu ile uygulanmistir. Bu, dizi boyutunda bir "
        "Python for-dongusu icerir; bu nedenle Tablo T4'teki gecikme rakamlari resmi "
        "'mamba-ssm' paketinin paralel-tarama CUDA cekirdekleriyle elde edecegi "
        "verimlilikten FARKLI olabilir -- bu, makalede acikca bir sinirlama olarak belirtilmelidir.",
        "- FGSM/PGD saldirilari, tablo verisinin gercek dagilim sinirlarina (StandardScaler "
        "sonrasi gozlenen min/maks) gore ozel olarak (Linf-norm, veri-dagilimina-duyarli "
        "clip ile) yeniden uygulanmistir.",
        "- Random Forest, gradyan-tabanli (FGSM/PGD) saldirilara turevlenemedigi icin dahil "
        "edilmemis, sadece kara-kutu transfer saldirisiyla degerlendirilmistir; ayni sebeple "
        "adversarial training savunmasi da RandomForest'a UYGULANAMAMAKTADIR.",
        f"- Tum sonuclar {len(CONFIG['seeds'])} rastgele tohum uzerinden ortalanmis; Mamba ile "
        "en guclu rakip arasindaki farkin istatistiksel anlamliligi esli t-testi ve Wilcoxon "
        "testi ile raporlanmistir (bkz. T2_istatistiksel_anlamlilik tablosu).",
        "- Saldiri alt-turu (attack_category) kirilganlik analizi, NSL-KDD icin standart "
        "DoS/Probe/R2L/U2R ust-kategori eslemesini (Tavallaee ve ark., 2009), NF-UNSW-NB15 "
        "ve CICIDS2018 icin veri setinin kendi orijinal saldiri etiketlerini kullanir.",
        "- Hiperparametre adaleti: ana deneyde tum mimariler AYNI d_model/n_layers degerlerini "
        "kullanir; bu tercihin sonuclari onyargilamadigini dogrulamak icin ayrica kucuk "
        "capli bir duyarlilik ablasyonu yapilmistir (bkz. T6, mevcutsa).",
    ]

    report_text = "\n".join(lines)
    with open(report_path, "w") as f:
        f.write(report_text)
    print(f"[RAPOR KAYDEDILDI] {report_path}")
    return report_path, report_text

print("Rapor uretim fonksiyonu hazir: generate_final_report(results_df, param_counts, defense_df=None, ablation_df=None, efficiency_df=None)")


## 17. ÇALIŞTIR: Tam Deney Pipeline'ı (Run All giriş noktası)

Bu hücre tüm pipeline'ı (ana deney + savunma değerlendirmesi + ablasyon)
**kapsamlı istisna yakalama** ve **e-posta bildirimi** ile çalıştırır.


In [ ]:
# ============================================================
# HÜCRE 17 — UST DUZEY CALISTIRMA (Run All giris noktasi)
# Bu, notebook'un SON ve TEK "buyuk" yurutme hucresidir. Tum pipeline
# burada, KAPSAMLI istisna yakalama ve e-posta bildirimi ile calisir.
#
# GUVENLIK/KESINTISIZLIK NOTU: Bu blok, calisma icinde olusan TUM
# Python istisnalarini yakalayip e-posta ile bildirir ve hatayi Drive'a
# loglar. Ancak Colab RUNTIME'IN TAMAMEN KOPMASI (internet kesintisi,
# GPU'nun elden alinmasi, tarayici sekmesinin kapanmasi vb.) durumunda,
# tanim geregi NOTEBOOK ICINDE calisan hicbir kod bunu tespit edip
# e-posta gonderemez (calisan surec sonlanmis olur). Bu riske karsi
# ana savunma hatti CHECKPOINT/RESUME mekanizmasidir (Hucre 9 ve 14):
# 'Run All' tekrar calistirildiginda TAMAMLANMIS isler ATLANIR ve
# kaldigi yerden devam edilir.
# ============================================================
import time

PIPELINE_START = time.time()

try:
    notify("PIPELINE BASLADI",
           f"Tam deney pipeline'i baslatildi. Veri setleri={CONFIG['datasets']}, "
           f"modeller={CONFIG['models']}, tohumlar={CONFIG['seeds']}.")

    results_df, param_counts = run_full_pipeline()

    defense_df = run_adversarial_training_defense(results_df)

    ablation_df = run_hyperparameter_ablation()

    generated_figures, generated_tables, efficiency_df = generate_all_outputs(
        results_df, param_counts, defense_df=defense_df, ablation_df=ablation_df
    )

    report_path, report_text = generate_final_report(
        results_df, param_counts, defense_df=defense_df, ablation_df=ablation_df, efficiency_df=efficiency_df
    )

    elapsed_min = (time.time() - PIPELINE_START) / 60.0
    summary_msg = (
        f"Pipeline BASARIYLA tamamlandi.\n\n"
        f"Toplam sure: {elapsed_min:.1f} dakika\n"
        f"Toplam sonuc satiri: {len(results_df)}\n"
        f"Uretilen grafik sayisi: {len(generated_figures)}\n"
        f"Uretilen tablo sayisi: {len(generated_tables)}\n\n"
        f"--- OZET RAPOR ---\n{report_text[:2500]}\n\n"
        f"Tum ciktilar Google Drive'da:\n{PATHS['base']}"
    )
    print("\n" + "=" * 70)
    print(summary_msg)
    print("=" * 70)

    notify("PIPELINE TAMAMLANDI (BASARILI)", summary_msg, attachment_path=report_path)

except Exception as e:
    elapsed_min = (time.time() - PIPELINE_START) / 60.0
    import traceback
    tb_text = traceback.format_exc()
    crash_log = f"{PATHS['logs']}/CRASH_{RUN_ID}.log"
    with open(crash_log, "w") as f:
        f.write(tb_text)
    notify(
        "PIPELINE HATA ILE DURDU",
        f"Pipeline {elapsed_min:.1f} dakika sonra HATA ile durdu.\n\n"
        f"Hata: {type(e).__name__}: {e}\n\n"
        f"Traceback (son 3000 karakter):\n{tb_text[-3000:]}\n\n"
        f"'Run All' tekrar calistirildiginda checkpoint'ler sayesinde kaldigi "
        f"yerden devam edilecektir.",
        attachment_path=crash_log,
    )
    raise


---
## Ekler / Notlar

- **Yeniden üretilebilirlik:** Tüm rastgelelik kaynakları (`random`, `numpy`, `torch`)
  `CONFIG["seeds"]` listesindeki 7 tohumla sabitlenmiştir; sonuçlar bu tohumlar
  üzerinden ortalanarak raporlanır (istatistiksel testler için yeterli güç).
- **Sınırlamalar:** Bu kod, Colab oturumunun **tamamen kopması** (ör. internet
  kesintisi) durumunu iç gözlemle tespit edip e-posta gönderemez — bu, tanım
  gereği çalışan sürecin sonlanması anlamına gelir. Bu riske karşı birincil
  savunma checkpoint/resume mekanizmasıdır.
- **Lisans/atıf:** NF-UNSW-NB15 kullanılıyorsa Sarhan, Layeghy ve Portmann'ın
  ilgili makalelerine; NSL-KDD kullanılıyorsa Tavallaee ve ark. (2009)'a
  (DoS/Probe/R2L/U2R kategori eşlemesi de bu makaleden alınmıştır);
  CICIDS2018 kullanılıyorsa Sharafaldin ve ark. (2018)'e atıf yapılmalıdır.
